# 11. The optimiser: the portfolio that tracks the benchmark most closely under constraints

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Risk-free rate | the return on a one-month US Treasury bill, the closest thing to a return with no risk |
| Excess return | a return minus the risk-free rate over the same period |
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money |
| Cap-weighted | weighted by market capitalisation |
| Benchmark | in the extension, the index the portfolio tracks, the cap-weighted combination of the 49 industries built in notebook 08 |
| Variance | the average squared distance of a series from its own average; its square root is the standard deviation, the usual measure of how much a return moves; volatility is the standard deviation of returns, stated per year here by multiplying the monthly figure by the square root of 12 |
| Covariance | a number that says how two series move together, positive when both tend to be above their averages in the same periods |
| Covariance matrix | the table of all variances and covariances of a set of assets, 49 by 49 here, 1,225 distinct numbers |
| Covariance estimator | a method for estimating the covariance matrix from a window of data; notebook 10 built four |
| Estimation window | the past returns an estimator sees, 120 months or three years of trading days here |
| Estimation error | the difference between a number estimated from a window and its true value |
| Shrinkage | pulling an estimate part of the way towards a simpler target, which trades a little bias for less estimation error; the shrinkage intensity is the fraction of the way it is pulled |
| Factor | a return series that moves many assets at once, for example the return of the whole market; the Fama-French factors are long-short portfolios built to isolate one such source each |
| Beta | how much an asset moves with a factor on average; the beta to the benchmark is how much it moves with the benchmark, one for the benchmark itself |
| Principal component | a combination of the assets, with one weight per asset, chosen so that it explains as much of the assets' total variance as a single combination can (notebook 09) |
| Active weight | the portfolio's weight in an asset minus the benchmark's weight in it |
| Tracking error | the standard deviation of the difference between two return series, stated per year; forecast (ex-ante) when computed from a covariance matrix before the month, realised (ex-post) when measured on the returns afterwards |
| Long-only | said of a portfolio in which no weight is negative, so nothing is sold that is not held |
| One-way turnover | half the sum over industries of the absolute changes in weight at a rebalance; the fraction of the portfolio sold, which is also the fraction bought when the portfolio stays fully invested; 0.02 means 2% of the portfolio changes hands |
| Turnover cap | an upper limit on the one-way turnover of a rebalance; 2% a month here |
| Drifted weights | the previous month's weights after that month's returns have moved them: each weight times one plus its industry's return, divided by the portfolio's own gross return, so they still sum to one |
| Tilt | a constraint that holds named industries below their benchmark weight; here Coal, Oil and Utilities at no more than half of it, a 50% reduction relative to the benchmark |
| Exclusion | a constraint that holds named industries at zero weight; in the mandate's set it holds Smoke (tobacco) and Guns (weapons) at zero |
| Beta-neutral | a constraint that holds the portfolio's beta to the benchmark equal to the benchmark's own, one; written as the active weights' beta summing to zero |
| Mandate | the whole set of rules a fund must obey; here the carbon tilt together with the exclusions of tobacco and weapons and beta neutrality |
| Robust portfolio | the weights whose largest forecast tracking error across a set of covariance matrices, the five estimators' here, is smallest; a portfolio that trusts no single matrix |
| Optimiser | the extension's routine that solves the constrained tracking-error problem: the weights that minimise forecast tracking error under a named set of constraints |
| Constraint set | one of the four cumulative sets C0 to C3 of the design: long-only; plus the active weight bound; plus the turnover cap; plus the mandate (the tilt alone in the design of 10 September; the tilt with beta neutrality and the exclusions from 4 October) |
| Feasible | said of weights that satisfy every constraint of the set; a set is infeasible in a month when no weights satisfy all of them at once |
| Binding | said of a constraint that holds with equality at the solution, so that relaxing it would lower the tracking error; a constraint that is not binding could be dropped without changing the answer |
| Quadratic program | a minimisation whose objective is a quadratic function of the weights and whose constraints are linear; when the quadratic is a variance it is convex and has one global minimum, which a solver finds exactly |
| Solver | the numerical routine that finds the minimum of a quadratic program; cvxpy with Clarabel here |
| Effective number of holdings | one divided by the sum of squared weights; 49 for 1/N across 49 industries, 1 for a portfolio in one industry, smaller the more concentrated the portfolio |
| Basis point | one hundredth of a percentage point, so 50 basis points is 0.50% |
| Sharpe ratio | the average monthly excess return divided by the standard deviation of the excess return; the reward earned per unit of risk taken |
| Active return | the portfolio's return minus the benchmark's return in the same period |
| Attribution | splitting a portfolio's active return into the parts due to named sources, the Fama-French factors in notebook 13, plus a remainder |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |

**The problem, stated the way practitioners state it.** Dom, Howard, Jansen and Lohre (2024), whose framing this notebook follows, argue that covariance estimators should be judged on portfolios a practitioner would hold: long-only, with bounds on weights, charged for trading, after costs. Their test portfolio is the constrained minimum-variance portfolio. An index manager's portfolio is the same problem with a different target: in place of the lowest variance, the smallest deviation from a benchmark, measured as tracking error, under the same kind of constraints and one more, the mandate's own rules about what may and may not be held. This notebook builds that portfolio. It is the point where the covariance stops being a forecast and becomes a decision: the optimiser takes a covariance matrix and a set of rules and returns the weights, and every rule's cost can be read off in forecast tracking error, which is how enhanced-index managers quote it.

**What this notebook does.** The replication asked which rule earns the best Sharpe ratio. The extension asks the question an index fund with a mandate asks: given a benchmark to track and a set of rules the fund must obey, find the weights that track the benchmark most closely. That is a minimisation. For a month with benchmark weights b and a covariance matrix Sigma, the optimiser finds the weights w that minimise the forecast tracking error, the square root of 12 times (w - b)' Sigma (w - b), subject to the constraints of the chosen set. With no constraints the minimum is at w = b and the tracking error is zero: holding the index tracks the index. Each constraint takes something away from that answer, and what it takes away is measured in forecast tracking error, which is how index managers quote the cost of a constraint. This notebook builds the optimiser (`src/bp/optimiser.py`), checks it against answers known by hand and against an independent solver (`tests/test_optimiser.py`), and runs it at three fixed months with each of notebook 10's estimators under each constraint set, to see what each constraint costs, where the tilt's weight goes, and how the turnover cap acts as a speed limit. Notebook 12 runs it in every month and judges the result on realised tracking error.

**The constraints, each with a number.**

- Budget: the weights sum to one. Always on; the benchmark's own weights sum to one.
- Long-only (C0): no weight below zero. The fund does not sell what it does not hold.
- Active weight bound (C1): each untilted industry within 2 percentage points of its benchmark weight. If Banks is 8.0% of the benchmark, the portfolio holds between 6.0% and 10.0% in Banks. The bound is around the benchmark weight because the 49 weights run from under 0.5% to over 10%, so one cap for all would be loose for the small industries and tight for the large ones.
- Turnover cap (C2): one-way turnover at most 2% a month, measured against the drifted weights. Selling 2% of the portfolio and buying 2% in one month is about 24% a year, the range of enhanced indexing. The fund starts as the index, so the cap applies from the first rebalance.
- Tilt (C3): Coal, Oil and Utilities each at no more than half their benchmark weight. If Oil is 3.0% of the benchmark, the portfolio holds at most 1.5%. The active weight bound does not apply to a tilted industry, because for any industry above 4% of the benchmark the two would contradict each other.
- Exclusion: named industries at zero. Part of the mandate's C3 (the second part of this notebook), for Smoke (tobacco) and Guns (weapons), the standard exclusions of Dutch institutional mandates.
- Beta-neutral: the portfolio's beta to the benchmark equal to one. Part of the mandate's C3 from the second part of this notebook on, after the first part showed that the tilt's hedge left a beta bet standing when the covariance was noisy.

**How this notebook is organised, and the two versions of C3.** The notebook has three parts, and the order is the order in which the work was done, because every expectation was written into `constants.py` before the code that tests it, and that order is the record. The first part runs the optimiser at three months under the constraint sets as first designed (10 September 2026), in which C3 is the tilt alone: long-only, the bound, the cap and the tilt. Expectations 1 to 5 belong to it. The second part is four checks on the practical problem behind the tilt's hedge (D1 to D4, 4 October), which found three problems a manager would fix, and the three fixes (F1 to F3, the same day): C3 became the mandate, the tilt with beta neutrality and the exclusions, and a robust portfolio joined the variants. The mandate's C3 is the one notebook 12 runs; the tilt-only set is kept in `constants.py` as `CONSTRAINT_SET_TILT_ONLY` so that the cost of the fixes can be measured there. The third part is one check on the solver itself (D5, 6 October), prompted by a month of notebook 12's first run.

**What the optimiser does with an infeasible month.** The tilt, the active bound and long-only are hard: a month in which they contradict each other stops the notebook. The turnover cap can make a month infeasible on its own, in the first month under a tilt (the previous portfolio is the benchmark and the tilt alone demands more trading than 2%) or in a month when the benchmark's weights moved by more than the cap. The optimiser therefore finds the smallest feasible one-way turnover first, by minimising turnover subject to the other constraints; when it exceeds the cap, the cap is raised to it, the month is counted, and only then is the tracking-error problem posed, so that the solver is never handed a problem that has no solution. Notebook 12 reports the count. The third part of this notebook (D5) shows what a solver does with a problem that has no solution, and why the order of the two steps matters.

**Where the numbers come from, and why this solver.** The constraints date from the design of 10 September 2026 and are in `constants.py` with their reasons.

- Tracking error. Public descriptions of enhanced indexing give the tracking error they run at: Robeco's enhanced-index developed-market composite ran at 1.06% a year from 2004 to 2016 and is offered up to about 5% (Robeco, "Building customized core quant portfolios", interview, October 2016); J.P. Morgan writes that enhanced index strategies typically have a tracking error below 2%, its own Research Enhanced Index strategy 0.67% over twenty years (J.P. Morgan Asset Management, "Celebrating 20 years of our Global Research Enhanced Index Strategy", 2024); Morgan Stanley's Applied Enhanced Index Russell 1000 strategy targets 1.5% to 2.0% with 200 to 300 holdings (strategy profile). The level of expectation 3, 1% a year, sits inside that range.
- Tilt. Robeco's enhanced-index products carry at least a 30% lower carbon footprint than the index, and its Paris-aligned versions target a 50% reduction (robeco.com, Enhanced Indexing Equities product page, read 4 October 2026). French's data carry no footprint per firm, so the tilt applies that 50% reduction to the three most carbon-intensive industries of the 49: Coal, Oil and Utilities.
- Active weight bound and turnover cap. The same descriptions call deviations from the index limited and turnover low, without a number. The 2 percentage points around each benchmark weight and the 2% one way a month (about 24% a year) are this study's own settings, chosen so that each constraint can bind without preventing the portfolio from following the benchmark; the tables report what each costs, which is the test of whether they are sensible.
- Exclusion. Built for the planned second study, what exclusion lists cost a long-only portfolio; the lists would come from what Dutch institutional investors publish about the firms they exclude.
- Solver. The problem is a convex quadratic program, and cvxpy is the layer that writes it almost as on paper and hands it to a solver (Diamond and Boyd, "CVXPY: A Python-Embedded Modeling Language for Convex Optimization", Journal of Machine Learning Research 17, 2016). It is the tool of the convex-optimisation portfolio literature: Boyd, Busseti, Diamond, Kahn, Koh, Nystrup and Speth, "Multi-Period Trading via Convex Optimization" (Foundations and Trends in Optimization, 2017), with three authors at BlackRock, ships with cvxportfolio, built on cvxpy. Asset managers run the same formulations in production on commercial solvers (MSCI's Barra Optimizer; Axioma, now part of SimCorp; Gurobi, which Robeco uses for its systematic fixed-income portfolios of about EUR 12.5 billion across about 30,000 instruments, Gurobi case study); what carries over from here to there is the formulation. The solver under cvxpy here is Clarabel (Goulart and Chen, 2024), an interior-point method, with its stopping rule tightened from the default.

**What I expect to see, written before the run (`constants.py`, 4 October 2026).**

1. Count first: the benchmark weights sum to one in every month from 1969-07; the drifted weights are defined for every month after the first and sum to one; Coal, Oil and Util are columns of the 49-industry file; the specification count is 28 (five estimator variants, the robust portfolio and RiskMetrics, times four constraint sets).
2. Known answers, at every report month and estimator: (a) under C0 and C1, with the benchmark as the previous portfolio, the solution is the benchmark and the forecast tracking error is zero within 1e-6 a year; (b) under C3 the three tilted industries sit at the top of their allowed range, half their benchmark weight, and the forecast tracking error is positive; (c) the forecast tracking error does not fall from C0 to C1 to C2 (each set adds a constraint, so the minimum cannot fall; this checks the solver) and rises from C2 to C3 (the tilt forces deviations that the freedom it gives the tilted industries does not repay; this is a statement about the data).
3. The cost of the tilt: at the last month, under C3 with the sample covariance on daily data, the forecast tracking error is below 100 basis points a year, the ex-ante tracking-error limit of enhanced indexing. Reason: the tilt removes about 3% of the portfolio from three industries whose own risk beyond the market is 12% to 20% a year, and the optimiser places that weight in the industries that move most with them, so the deviation's risk is a fraction of 3% times 20%.
4. Reported, no verdict: where the tilt's weight goes (the industries with the largest positive active weights under C3), the effective number of holdings of the benchmark and of each portfolio, the forecast tracking error under C3 across the five estimators (notebook 12 judges them on realised tracking error), and which constraints bind.
5. The turnover cap as a speed limit, a stress case at the last month: starting from 1/N weights under long-only plus the cap, the one-way turnover equals the cap (it binds) and the forecast tracking error is positive, while under C1 from the same start it is zero. Reported, no verdict: the forecast tracking error after 1, 6, 12 and 24 monthly rebalances from 1/N with the benchmark and the covariance held fixed; the one-way distance from 1/N to the benchmark divided by the cap; and what C2 does from the same start, where the active bound is hard and demands more trading than the cap allows, so the cap is relaxed to the smallest feasible turnover.

**Checks and fixes added after the first part, each with its expectation written in `constants.py` before its code.** D1 to D4 (4 October): how much of the tilt is a market bet, what forcing the beta to one costs, what refusing the tobacco hedge costs, and how much the hedge depends on the estimator and the month. F1 to F3 (4 October): the mandate's C3 and the robust portfolio at the same three months. D5 (6 October): the solver's status against the independent check on a month with no feasible portfolio. Their expectations are stated in full at the head of their sections.

Running time: about three minutes, most of it the download of the daily files.

## Modules

`constants`, `french_loader`, `benchmark`, `rules` and `covariance` are those of earlier notebooks; `covariance` gained the panel loader and the estimator dispatch that notebook 10's loop used inline. `optimiser` is new: the constrained tracking-error minimiser, the drifted weights, the feasibility checks and the effective number of holdings, each tested on made-up markets in `tests/test_optimiser.py`, including against an independent solver. `evaluation`, notebook 12's module, is written here as well, for the check D5, which reruns the first five years of one of notebook 12's paths.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("cvxpy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cvxpy"])
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) and the
design of the extension (section 5) were fixed on 10 September 2026, before any
of the rules' code was written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the extension's covariance estimators (section 5).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks): the cap-weighted combination
# of the 49 industries, weights from firm count times average firm size, must
# track the market factor plus the risk-free rate within this tracking error a
# year over the full sample (notebook 08). If it does not, the derived weights
# are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Notebook 08, the cap-weight check, fixed 1 October 2026 before its code. The
# weight of industry i in month t is its firm count times its average firm size
# in French's file for month t, divided by the sum over the 49 industries;
# French measures both at the start of month t, so the weight uses nothing
# unknown when the month begins. The previous-month reading is run beside it as
# a check of this timing, and the notebook stops if it tracks more closely.
CAPW_WEIGHT_TIMING = "same_month"
CAPW_WEIGHT_TIMING_ALTERNATIVE = "previous_month"   # reported beside the chosen timing, never used as the benchmark
CAPW_MARKET_SERIES = ("Mkt-RF", "RF")                # the market series is Mkt-RF plus RF, so that both sides are plain returns
CAPW_REPORT_MONTHS = ("first", "2004-11", "last")

# The direct test of where the benchmark's gap to the market comes from
# (notebook 08, second part), fixed 1 October 2026 before its code. French forms
# the industry portfolios once a year at the end of June from the firms that
# have an SIC code then, while the market return takes in a firm from its first
# full month of trading, so between one July and the next the market holds
# firms that no industry portfolio holds. The test needs the firm list of each
# portfolio each month, which French does not publish, so it runs on the CRSP
# monthly stock file (WRDS, under Queen Mary University of London's
# subscription), the one use of data outside French's library in the study.
# The CRSP data stay on the machine that holds them: no notebook reads them,
# the repository carries none, and only the aggregates in
# DIRECT_TEST_RESULTS_FILE are published, with CRSP cited by name. The market
# universe is CRSP share codes 10 and 11 on NYSE, AMEX and NASDAQ, weighted by
# start-of-month market equity; the June-formed universe is the firms of that
# market that were in it at the end of June of the formation year with a
# positive SIC code. The script is checks/crsp_direct_test.py; its pure
# functions are in bp.direct_test.
DIRECT_TEST_SHARE_CODES = (10, 11)
DIRECT_TEST_EXCHANGE_CODES = (1, 2, 3)
DIRECT_TEST_FORMATION_MONTH = 6          # the end of June
DIRECT_TEST_MARKET_TE_MAX = 0.0020       # expectation 1: the CRSP market universe tracks French's market return within 20 basis points a year
DIRECT_TEST_MIN_CORRELATION = 0.80       # expectation 2: correlation of the predicted gap with the observed one
DIRECT_TEST_SLOPE_RANGE = (0.8, 1.2)     # expectation 2: slope of the observed gap on the predicted one
DIRECT_TEST_RESIDUAL_TE_MAX = 0.0025     # expectation 4: tracking error of what the explanation leaves, 25 basis points a year
DIRECT_TEST_RESULTS_FILE = "checks/results_crsp_direct_test.json"   # aggregates only; committed

# Constraints. Fixed 10 September 2026; the sources for the levels are in
# notebook 11, "Where the numbers come from".
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order. C3 is the mandate: the
# tilt with beta neutrality and the exclusions. Beta neutrality and the
# exclusions were added on 4 October 2026, after notebook 11's checks D1 to D4
# and before notebook 12's code, because the tilt's hedge left a beta bet
# standing when the covariance was noisy and bought tobacco and weapons. The
# tilt alone is CONSTRAINT_SET_TILT_ONLY, the C3 of the first design, reported
# beside the mandate in notebooks 11 and 12 so that the cost of the two
# additions is visible.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion"),
}
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the C3 of the design of 10 September 2026
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, the standard exclusions of Dutch institutional mandates, in French's industry names; weight zero

# The data the covariance estimators see, fixed 1 October 2026. The 120-month
# window is the paper's convention and gives 49 x 120 = 5,880 returns for the
# 1,225 entries of a 49-industry covariance matrix, fewer than five per entry.
# Practitioners estimate from daily returns (three years hold about 756 trading
# days, 30 observations per entry), impose structure, and weight recent days
# more; the extension does all three. Each estimator is built at two data
# frequencies: "monthly_120", the last EXT_ESTIMATION_WINDOW months of monthly
# excess returns, and "daily_3y", the last EXT_COV_DAILY_WINDOW_YEARS years of
# daily excess returns scaled to a monthly covariance by DAILY_TO_MONTHLY_SCALE
# (notebook 10 tests that scaling). A trading day on which any industry has no
# return (one in the sample, Softw on 1971-03-11) is left out of the window.
EXT_COV_FREQUENCIES = ("monthly_120", "daily_3y")
EXT_COV_DAILY_WINDOW_YEARS = 3
EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY = (1, 5)   # reported for the sample estimator only, no verdict
DAILY_TO_MONTHLY_SCALE = 21                        # 252 trading days a year divided by 12
EXT_COV_EWMA_HALFLIFE_DAYS = 252                   # the recency-weighted sensitivity: a half-life of one year of trading days, the RiskMetrics setting of Dom, Howard, Jansen and Lohre (2024, section 3.3.2); 126 until 3 October 2026, changed on reading that paper, before notebook 10's code

# Covariance estimators. The shrinkage target follows Dom, Howard, Jansen and
# Lohre (2024), the paper the design named as its reference, checked on
# 3 October 2026 before notebook 10's code: their linear shrinkage pulls the
# sample covariance towards a scaled identity matrix (Ledoit and Wolf 2004b),
# and their footnote 11 reports that the constant-correlation target gave
# inferior results. The same paper estimates from three years of daily returns
# and rebalances monthly, and finds that under long-only constraints the
# differences between estimators shrink, which is notebook 10's expectation 3.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "scaled_identity"          # Ledoit and Wolf (2004b), "A well-conditioned estimator for large-dimensional covariance matrices"; was "constant_correlation" until 3 October 2026
LW_TARGET_SENSITIVITY = "constant_correlation"   # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"; reported, no verdict
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # notebook 09, expectation 2: the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Notebook 09, the principal components of the 49 industries, fixed 1 October
# 2026 before its code; the expectations are in its introduction.
PCA_REPORT_COMPONENTS = 10
PCA_PARALLEL_DRAWS = 200
PCA_PARALLEL_PERCENTILE = 95
PCA_PARALLEL_SEED = 20260910
# Set after notebook 09 ran, as its design said: the first component's
# correlation with the benchmark fell below PCA_PC1_MARKET_CORR_MIN in the
# window from 2023 (0.837), so notebook 10's PCA estimator takes the
# benchmark's excess return as its first factor. Parallel analysis found two
# components above noise; PCA_N_COMPONENTS stays at five and the two-component
# version is reported beside it.
PCA_FIRST_COMPONENT = "market"        # notebook 10: the benchmark's excess return is the first factor; "data" would take it from the eigenvectors
PCA_N_COMPONENTS_SENSITIVITY = 2      # the parallel-analysis count on the full sample, reported beside PCA_N_COMPONENTS

# Notebook 10, the four covariance estimators, fixed 3 October 2026 before its
# code. The estimators, data frequencies, tests and expectations are stated in
# the notebook's introduction and in covariance.py.
COV_EVAL_START = "1979-07"            # 1969-07 plus 120 months; the three-year daily window starts 1972-07 and is the shorter requirement
COV_TEST_ACTIVE_DRAWS = 200           # random active-weight vectors for the calibration test
COV_TEST_SEED = 20260910
COV_LONG_ONLY_RANGE_MAX = 0.01        # expectation 3: one percentage point of annualised volatility across the four estimators on daily data
COV_SCALE_RATIO_RANGE = (0.8, 1.25)   # expectation 4: the ratio of the benchmark's realised monthly variance to 21 times its daily variance
# Three tests added after notebook 10's first run (4 October 2026), each fixed
# here before its code and labelled in the notebook: P1, a scale correction by
# the variance ratio of Lo and MacKinlay (1988), one number from the benchmark's
# daily returns that multiplies the daily covariance in place of
# DAILY_TO_MONTHLY_SCALE alone; P2, bias statistics by decade; P3, trading costs
# of the minimum-variance portfolios at EXT_COST_LEVELS.
COV_SCALE_VR_HORIZON_DAYS = DAILY_TO_MONTHLY_SCALE   # the horizon of the variance ratio: one month of trading days
COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY = 10           # P1 sensitivity: the variance ratio from ten years of daily benchmark returns
COV_POST_RUN_DECADES_MIN_CLOSER = 4                  # P1a: of five decades, the corrected forecast is closer to calibrated in at least this many
COV_POST_RUN_COST_MIN_VARIANTS = 6                   # P3a and P3b: of eight main variants, the long-only portfolio beats the unconstrained one net of cost in at least this many

# Notebook 11, the optimiser, fixed 4 October 2026 before its code; the
# constraints themselves date from 10 September. The problem, the infeasible-month
# rule and the expectations are stated in the notebook's introduction and in
# optimiser.py.
OPT_SOLVER = "CLARABEL"                  # interior-point solver shipped with cvxpy; deterministic
OPT_SOLVER_OPTIONS = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-10}   # tighter than the solver's defaults (1e-8), which left weights 1e-4 off an independent solver's on a test problem
OPT_SOLVER_OPTIONS_ROBUST = {"tol_gap_abs": 1e-9, "tol_gap_rel": 1e-9, "tol_feas": 1e-9}   # the robust problem is written with second-order cones, on which the solver reaches 1e-9 and reports 1e-12 as inaccurate; the objective agrees with the 1e-12 run to 1e-10 on the test problems
OPT_TOL = 1e-6                           # feasibility checks on weights, bounds and turnover; also "binding" means within this of the limit
OPT_TOL_TE = 1e-6                        # expectation 2a: a forecast tracking error below this, per year, counts as zero
OPT_TILT_TE_MAX_ANNUAL = 0.01            # expectation 3: 100 basis points a year, the enhanced-indexing ex-ante limit (Robeco's composite ran at 1.06%, J.P. Morgan quotes below 2%; sources in notebook 11)
OPT_REPORT_MONTHS = ("first_eval", "2004-11", "last")   # first_eval is COV_EVAL_START, the first month with both windows
OPT_SPEED_LIMIT_STEPS = (1, 6, 12, 24)   # expectation 5: rebalances reported on the journey from 1/N to the benchmark
# Checks D1 to D4 on the practical problem behind the tilt's hedge, added after
# notebook 11's first run (4 October 2026), each fixed here before its code.
OPT_D1_MARKET_SHARE_MAX = 0.05         # D1a: the market part of the tilt's forecast active variance is below this share in every month-estimator pair
OPT_D2_TE_RISE_MAX = 0.0001            # D2a: a beta-neutral constraint raises the forecast tracking error by less than one basis point a year in every pair
OPT_D3_EXCLUDE = ("Smoke",)            # D3: the hedge's tobacco purchase refused, its cost reported
# The fixes F1 to F3 (4 October 2026, before notebook 12's code): beta neutrality
# and the exclusions join C3 (above), and a robust portfolio, the weights whose
# largest forecast active variance across the estimators' covariances is
# smallest (Goldfarb and Iyengar 2003; Tutuncu and Koenig 2004), joins the
# rolling evaluation as a variant. Its beta neutrality holds under the average
# of the five beta vectors, because one equality per covariance was infeasible
# at 1979-07 under the mandate's bounds.
OPT_FIX_RISE_MAX = 0.0015              # F2: the mandate's C3 costs less than 15 basis points a year more than the tilt alone
ROBUST_VARIANT = ("robust", "all_five")   # the robust portfolio: minimax over the five covariances
# Check D5 on the solver, added after notebook 12's first run (6 October 2026),
# fixed here before its code: at this month the index moved more than the cap
# allows, no portfolio satisfied the cap and the tilt at once, and the solver
# reported "optimal" with weights that fail the independent check. The
# optimiser therefore finds the smallest feasible turnover first and never
# poses a capped problem that has no solution; the check of every returned
# portfolio stays.
OPT_D5_MONTH = "1984-07"

# Notebook 12, the rolling evaluation, fixed 4 October 2026 before its code: one
# path per specification, each portfolio built on the data before its month,
# judged on realised tracking error, calibration of the forecasts, active
# return net of costs, turnover and holdings. The expectations are in the
# notebook's introduction.
EVAL_TE_RANGE_MAX = 0.0010           # expectation 2: 10 basis points a year across the four daily estimators
EVAL_BIAS_RANGE = (0.8, 1.25)        # expectation 3, the same band as COV_SCALE_RATIO_RANGE
EVAL_RELAX_SHARE_MAX = 0.10          # expectation 6: at most one month in ten with a relaxed cap
EVAL_START_PORTFOLIO = "benchmark"   # every path starts as the index
EVAL_BENCHMARK_TOL = 1e-6            # expectation 1: an active return below this a month counts as holding the benchmark; written as 1e-10 and set to the solver's precision (about 1e-7) before the recorded run
# Expectations 8 and 9, added after notebook 12's first run (6 October 2026),
# each fixed here before its code and labelled in the notebook. RiskMetrics
# joins as a seventh variant because Dom, Howard, Jansen and Lohre (2024) find
# that time dynamics matter more than shrinkage or factor structure; the
# calibration multiplies each forecast by the ratio of realised to forecast
# tracking error over the previous EVAL_CALIBRATION_WINDOW months of the same
# path, using only months before the forecast.
EWMA_VARIANT = ("sample", "daily_5y_ewma")   # the seventh variant: RiskMetrics, as in notebook 10's sensitivity
EVAL_CALIBRATION_WINDOW = 60                 # months of a path's own past the calibration uses; a standard deviation from 60 observations has a standard error of about 9%
EVAL_CALIBRATION_MIN = 24                    # the forecast is left as it is while fewer months are available
EVAL_CALIBRATED_BIAS_RANGE = (0.85, 1.15)    # expectation 9: the calibrated bias statistic under C3, four daily estimators

# Notebook 13, the factor attribution, fixed 6 October 2026 before its code: each
# path's monthly active return split into the contributions of the six factors
# (the active exposure to each, from the industries' betas estimated on the data
# before the month, times the factor's return in the month) plus a residual, the
# industries' own returns; the identity is the check. The paths attributed are
# the seven variants under the mandate's C3 and under the tilt alone, rebuilt
# from the modules; the expectations are in the notebook's introduction.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # expectation 1: contributions plus residual minus active return, per month, in return units
ATTRIBUTION_SETS = ("C3", "C3 tilt only")        # the two constraint sets whose paths are attributed
ATTRIBUTION_REPRODUCTION_TOL = 0.00001           # expectation 1: the rebuilt paths reproduce notebook 12's realised tracking errors within 0.1 basis point a year
ATTRIBUTION_REFERENCE_TE = {                     # notebook 12's realised tracking errors, per year, for the reproduction check
    ("sample, daily 3y", "C3"): 0.0084654, ("ledoit_wolf, daily 3y", "C3"): 0.0084820, ("factor, daily 3y", "C3"): 0.0093449,
    ("pca, daily 3y", "C3"): 0.0091482, ("sample, monthly 120", "C3"): 0.0095761, ("robust, all five", "C3"): 0.0087358,
    ("RiskMetrics, daily 5y", "C3"): 0.0084278,
    ("sample, daily 3y", "C3 tilt only"): 0.0080276, ("ledoit_wolf, daily 3y", "C3 tilt only"): 0.0080282, ("factor, daily 3y", "C3 tilt only"): 0.0088296,
    ("pca, daily 3y", "C3 tilt only"): 0.0086722, ("sample, monthly 120", "C3 tilt only"): 0.0090924, ("robust, all five", "C3 tilt only"): 0.0082305,
    ("RiskMetrics, daily 5y", "C3 tilt only"): 0.0079698,
}
ATTRIBUTION_MARKET_EXPOSURE_MAX = 0.05           # expectation 2: the mean active exposure to Mkt-RF, in absolute value, for every path
ATTRIBUTION_SIGN_AGREEMENT_MIN = 4               # expectation 3: of the five factors other than the market, the number whose mean active exposure under the mandate has the same sign for all seven variants
ATTRIBUTION_FACTOR_SHARE_MAX = 0.5               # expectation 4: the share of the active return's variance the six factors explain, for every path
ATTRIBUTION_FACTOR_PART_MAX = 0.5                # expectation 5a: the share of the mandate's cost against the tilt alone that comes through the factor contributions, every variant
ATTRIBUTION_ARITHMETIC_TOL = 0.0002              # expectation 5b: the exclusions' factor part (the five factors other than the market) within this of the arithmetic of the inputs, per year (2 basis points), every variant

# Specification count, written down before the code runs and reported.
EXT_SPEC_COUNT = (len(COV_ESTIMATORS) + 3) * len(CONSTRAINT_SETS)   # 28 optimised specifications: four daily-based estimators, the monthly sample covariance, the robust portfolio and RiskMetrics, times four constraint sets
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 7. Version 2, module A: the factor objective under the mandate
# ---------------------------------------------------------------------------
# Fixed 8 October 2026, before any of the module's code was written, from the
# design of 7 October 2026 and version 1's results at commit 6835d49. Version 1 found that
# the mandate's tilt leaves incidental exposures of 0.004 to 0.018 in absolute
# value to the six factors, costing 9 to 13 basis points a year. Module A makes
# the exposures the objective: each month, with the data available before the
# month, the optimiser maximises the sum of the active exposures to the four
# targeted factors, s' B' (w - b), where B holds the industries' betas to the
# six factors estimated as in notebook 13 (three years of daily returns before
# the month) and s is FO_TARGET_SIGN, under every constraint of the mandate
# (CONSTRAINT_SETS["C3"]) and one new constraint, the budget: the forecast
# tracking error from the month's covariance estimate at most FO_BUDGETS_ANNUAL.
# The study holds no view on expected returns, as version 1 did not: the
# targets are the four of French's six factors with a documented premium, the
# direction of each is the literature's, and the four are weighted equally in
# beta units. Nothing estimates a premium or times a factor.
# Naming, fixed in the design: "tilt" stays the industry reduction of version 1;
# the new objective is the factor objective, the four are the targeted factors,
# the limit is the budget.
FO_DECISION_DATE = "2026-10-08"
FO_TARGET_FACTORS = ("HML", "RMW", "CMA", "Mom")              # value, profitability, investment, momentum
FO_TARGET_SIGN = {"Mkt-RF": 0.0, "SMB": 0.0, "HML": 1.0, "RMW": 1.0, "CMA": 1.0, "Mom": 1.0}   # s, on ATTRIBUTION_FACTORS; +1 is the direction of the documented premium
FO_CONSTRAINT_SET = "C3"                                       # the mandate: every constraint of version 1's C3, unchanged
FO_BUDGETS_ANNUAL = (0.0075, 0.0100, 0.0150)                   # tau, per year: just above the mandate's own realised cost (84.7 to 95.8 bp), the enhanced-indexing limit of OPT_TILT_TE_MAX_ANNUAL, and the generous case
FO_BUDGET_MAIN = 0.0100                                        # the budget at which the robust, RiskMetrics and single-factor paths run and most expectations are tested
FO_BETA_WINDOW_YEARS = EXT_COV_DAILY_WINDOW_YEARS              # the betas' window: notebook 13's three years of daily returns before the month
# Infeasibility rule: in a month in which the budget cannot be met under the
# mandate (the tracking-error-minimising solution of version 1 already exceeds
# tau), the path takes that minimising solution, the month is counted and
# reported, and nothing else is relaxed; the count is one of the results.
FO_INFEASIBLE_RULE = "tracking-error minimiser"
# The 19 paths, the count fixed here: the five estimator variants of version 1
# at each of the three budgets (15); the robust and RiskMetrics variants at the
# main budget (2); two single-factor sensitivities at the main budget with the
# daily sample estimator, s on value alone and on momentum alone (2). Version
# 1's mandate paths (tracking error minimised, no objective) are the comparison
# and are not rerun. A path added after the first run is disclosed with its
# date and reason, and the count restated.
FO_SINGLE_FACTOR_PATHS = (("HML",), ("Mom",))
FO_PATH_COUNT = 19
# The two measures new to module A. The exposure the budget bought: the mean
# over the months of the sum of the four targeted active exposures, divided by
# the realised tracking error in basis points a year. The gap: for each month,
# the targeted exposure the optimiser chose ex ante, s' B_t' a_t with B_t the
# betas estimated before the month, minus the exposure the same weights carry
# when the betas are estimated after the month, s' B_{t+1}' a_t; its mean
# absolute value over the months is the path's gap.
FO_GAP_DEFINITION = "s'(B_t - B_{t+1})' a_t, mean absolute value over the months"
# The seven expectations, each a strict pass or fail; the levels from
# version 1's results at 6835d49, with the design's reasons.
# E1 and E2 were restated on 9 October 2026, after notebook 14's check of
# the arithmetic and before any path ran, by the design of 9 October 2026:
# a level changed at that point, between the pre-run check notebook 14
# exists for and the first rolling run, is a pre-run restatement, dated and
# disclosed here and once in the write-up; a level changed after notebook 15
# runs is a post-run change. The original levels of 8 October stay below,
# marked as superseded.
FO_RESTATEMENT_DATE = "2026-10-09"
# E1. At the main budget the budget binds (forecast tracking error within
# FO_E1_BIND_TOL of the budget) in at least FO_E1_BIND_SHARE_MIN of the
# months in which the mandate's own minimum forecast tracking error, with
# turnover free, is at or below the budget, for each of the four daily
# estimators. The months the mandate alone cannot meet are reported per
# estimator and budget from notebook 14's feasibility table; the months the
# path misses beyond them, with the turnover cap in force, are counted and
# reported, since that difference is what the path adds. Reason for the
# restatement: notebook 14 found the mandate alone above 100 bp in 37
# (sample), 43 (Ledoit-Wolf) and 39 (six-factor) of the 566 months, 22 for
# principal components, nearly all in 1979 to 1989, so the budget cannot
# bind in those months whatever the objective.
FO_E1_BIND_TOL_ANNUAL = 0.0001           # one basis point a year
FO_E1_BIND_SHARE_MIN = 0.95
FO_E1_DENOMINATOR = "months the mandate alone can meet"
# Superseded on 9 October 2026: the same share of all 566 months. Reason
# given on 8 October: version 1's forecast under the mandate was 55 to 68 bp
# on average and 47 to 55 for the tilt alone at the 1979 oil peak for the
# daily estimators, so a budget of 100 should be reachable almost always;
# the monthly estimator (108 at that peak) may fail it in the first years.
FO_E1_DENOMINATOR_SUPERSEDED_2026_10_08 = "all 566 months"
# E2. At the main budget the mean active exposure to each targeted factor is
# positive on every main path, the average of the four is at least
# FO_E2_MEAN_EXPOSURE_MIN for the daily sample estimator, and that average
# rises with the budget (75, then 100, then 150) on every estimator; the four
# exposures are reported per path at every budget, the average never alone.
# Reason for the restatement: notebook 14's stationary mean of the four at
# 100 bp is 0.030 for the daily sample estimator and a capped path holds at
# most the stationary exposure on average; the level is four fifths of
# 0.030, the fifth being the design's allowance for what the turnover cap,
# the infeasible months and the rolling betas may cost against the
# stationary solution, a judgement stated as such. Under an active weight
# bound of 2 percentage points on 49 industries, 100 bp of budget buys about
# 0.03 of average exposure in beta units.
FO_E2_MEAN_EXPOSURE_MIN = 0.024
# Superseded on 9 October 2026. Reason given on 8 October: version 1's
# incidental exposures were 0.004 to 0.018 in absolute value with no
# objective; 49 industries within 2 percentage points of their weights can
# reach three to ten times that when the objective asks for it.
FO_E2_MEAN_EXPOSURE_MIN_SUPERSEDED_2026_10_08 = 0.05
# One diagnostic per path, counted and reported with no expectation
# attached (the design of 9 October 2026): the months in which any targeted
# exposure ends below its incidental value, the value the mandate's
# tracking-error minimiser carries. Notebook 14 found such a month in 5 of
# its 15 month-estimator pairs at 100 bp.
FO_DIAGNOSTIC_BELOW_INCIDENTAL = "months with a targeted exposure below its incidental value"
# E3. Realised tracking error exceeds the budget by 20 to 60 per cent on the
# daily estimators, and by more on the monthly sample covariance; tested at
# every budget (the design names none), the monthly ratio against the largest
# daily ratio at the same budget. Reason: version 1's bias was 1.22 to 1.39 on
# daily data and 1.61 on monthly; a budget-binding solution concentrates risk
# where the estimator sees least, so the miss may exceed version 1's, bounded
# here at 1.6.
FO_E3_REALISED_OVER_BUDGET_RANGE = (1.2, 1.6)
# E4. The four targeted factors' contributions sum to a positive amount on
# every main path, and at the main budget and above the six factors explain at
# least FO_E4_FACTOR_SHARE_MIN of the active return's variance on every main
# path. Reason: version 1's factors explained 14 to 28 per cent of incidental
# exposures; deliberate exposures raise the share.
FO_E4_FACTOR_SHARE_MIN = 0.30
# E5. At the main budget the gross active return is positive for at least
# FO_E5_POSITIVE_MIN of the five estimators; the net return at DGU_TCOST per
# unit of turnover lies within FO_E5_NET_GAP_MAX of the gross. Whether the
# active return is more than two standard errors from zero is reported, not
# expected. Reason: the study holds no view on the premia; the sign follows
# from the targets if the premia had their documented sign over the sample,
# which is what the data say and not a claim of skill.
FO_E5_POSITIVE_MIN = 4
FO_E5_NET_GAP_MAX = 0.0010               # 10 basis points a year
# E6. Of the two single-factor paths, momentum alone has the higher turnover
# and the larger gap, and the turnover cap binds in more months on the
# momentum path than on the value path. Reason: industry momentum betas rotate
# within a year; value betas do not.
# E7. At the main budget the spread across the four daily estimators in
# realised tracking error is at most FO_E7_TE_RANGE_MAX, the exposure bought
# per basis point differs across them by less than FO_E7_EXPOSURE_PER_BP_REL_MAX
# of their mean (largest minus smallest over the mean), and the monthly sample
# covariance remains the worst calibrated (the largest realised over forecast
# of the five). Reason: version 1, a range of 9 basis points on a level of
# about 90, the monthly covariance 61 per cent under-forecast.
FO_E7_TE_RANGE_MAX = 0.0020              # 20 basis points a year
FO_E7_EXPOSURE_PER_BP_REL_MAX = 0.30
# Notebook 14 runs the objective at the report months of version 1 and over
# the sample without a rolling path: a worked month, the exposure against the
# budget at FO_CURVE_BUDGETS (the three budgets and points between), and the
# feasibility of each budget in every month, from the forecast tracking error
# of the tracking-error-minimising solution under the mandate.
FO_CURVE_BUDGETS_ANNUAL = (0.0050, 0.0075, 0.0100, 0.0125, 0.0150, 0.0200)
FO_TOL_TE = 1e-9                         # a forecast tracking error within this of the budget counts as on it, in the tests

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = chosen["frame"].copy()
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/bp/benchmark.py
"""The benchmark of the extension: the cap-weighted combination of French's 49 industry portfolios.

Each industry portfolio's value-weighted return for a month is the return of the
firms in the industry, each counted in proportion to its market equity at the
start of the month. Weighting the 49 industry returns by each industry's total
market equity at the same time (firm count times average firm size, two blocks of
the same French file) gives the value-weighted return of all the firms in the 49
portfolios, which should be close to French's market return. Notebook 08 measures
how close, against the level fixed in constants.CAPW_TE_MAX_ANNUAL, and decides
whether these weights are the benchmark of notebooks 09 to 13.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import french_loader as fl

PERIODS_PER_YEAR = 12


def load_inputs(cache_dir: str = "data/french") -> dict[str, pd.DataFrame | pd.Series]:
    """The three monthly blocks of the 49-industry file and the market series.

    returns: value-weighted industry returns (decimal); firms: number of firms;
    size: average firm size in millions of dollars; market: Mkt-RF plus RF from
    the factors file (decimal). Missing codes are NaN in all of them.
    """
    returns = fl.load_monthly("ind49", r"Value Weighted Returns -- Monthly", cache_dir)
    firms = fl.load_counts("ind49", r"Number of Firms", cache_dir)
    size = fl.load_counts("ind49", r"Average Firm Size", cache_dir)
    f3 = fl.load_monthly("factors3", 0, cache_dir)
    excess, rf = C.CAPW_MARKET_SERIES
    market = (f3[excess] + f3[rf]).rename("market")
    return {"returns": returns, "firms": firms, "size": size, "market": market}


def market_equity(firms: pd.DataFrame, size: pd.DataFrame) -> pd.DataFrame:
    """Total market equity of each industry, in the units of the size block (millions of dollars)."""
    if not firms.index.equals(size.index) or not firms.columns.equals(size.columns):
        raise ValueError("firm counts and average sizes must share months and industries")
    return firms * size


def cap_weights(firms: pd.DataFrame, size: pd.DataFrame, timing: str = C.CAPW_WEIGHT_TIMING) -> pd.DataFrame:
    """Each industry's share of total market equity, per month; the rows sum to one.

    timing "same_month": the count and size of month t weight the returns of
    month t (French measures both at the start of the month). "previous_month":
    the count and size of month t-1 weight the returns of month t; the first
    month has no weights. Any month in which an industry's count or size is
    missing has no weights at all, so that a partial sum never passes as a
    whole-market weight.
    """
    me = market_equity(firms, size)
    if timing == "previous_month":
        me = me.shift(1)
    elif timing != "same_month":
        raise ValueError(f"unknown timing {timing!r}")
    total = me.sum(axis=1, min_count=me.shape[1])
    return me.div(total, axis=0)


def combination_return(returns: pd.DataFrame, weights: pd.DataFrame) -> pd.Series:
    """Sum over industries of weight times return, per month; NaN where any of the 49 is missing."""
    if not returns.columns.equals(weights.columns):
        raise ValueError("returns and weights must have the same industries in the same order")
    aligned = weights.reindex(returns.index)
    return (aligned * returns).sum(axis=1, min_count=returns.shape[1]).rename("combination")


def tracking_error(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> float:
    """Annualised standard deviation of the per-period difference a - b, over the periods both have."""
    d = (a - b).dropna()
    if len(d) < 2:
        return float("nan")
    return float(d.std(ddof=1) * np.sqrt(periods_per_year))


def compare(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> dict[str, float]:
    """Tracking error, annualised mean difference, correlation, largest absolute monthly difference and month count of a against b."""
    both = pd.concat([a, b], axis=1).dropna()
    d = both.iloc[:, 0] - both.iloc[:, 1]
    return {
        "months": int(len(d)),
        "tracking_error_annual": float(d.std(ddof=1) * np.sqrt(periods_per_year)),
        "mean_difference_annual": float(d.mean() * periods_per_year),
        "correlation": float(both.iloc[:, 0].corr(both.iloc[:, 1])) if both.std(ddof=1).gt(0).all() else float("nan"),
        "max_abs_monthly_difference": float(d.abs().max()),
        "month_of_max": str(d.abs().idxmax()),
    }


def weight_extremes(weights: pd.DataFrame, month: str | pd.Period) -> dict[str, object]:
    """The largest and smallest weights in one month, with their industries."""
    row = weights.loc[pd.Period(month, "M")].dropna()
    return {
        "month": str(pd.Period(month, "M")),
        "largest": row.idxmax(), "largest_weight": float(row.max()),
        "smallest": row.idxmin(), "smallest_weight": float(row.min()),
        "industries": int(len(row)),
    }

In [ ]:
%%writefile src/bp/rules.py
"""
Portfolio rules from DeMiguel, Garlappi and Uppal (2009), section 1.

Each rule is a function of one estimation window of monthly excess returns,
an array of shape (M, N), and returns the vector of relative weights on the N
risky assets, DGU equation (1): the mean-variance solution x = Sigma^-1 mu / gamma
normalised by the absolute value of its sum, so that the weights sum to +1 or
to -1 and the direction of the position is preserved in the few windows where
the risky assets are net short.

Notation follows the paper: mu and Sigma are the sample moments of the window,
M its length, N the number of assets. The risk aversion gamma cancels in the
normalisation for every unconstrained rule, which is why it does not appear.

Rules in this module, with the DGU abbreviation and section:
  ew       1/N, section 1.1
  mv       sample-based mean-variance, section 1.2
  bs       Bayes-Stein shrinkage, section 1.3.2, Jorion (1986)
  min      minimum variance, section 1.4.1
  vw       the value-weighted market, section 1.4.2 (all weight on the market column)
  mv-c     short-sale-constrained mean-variance, section 1.5
  bs-c     short-sale-constrained Bayes-Stein, section 1.5
  min-c    short-sale-constrained minimum variance, section 1.5
  g-min-c  minimum variance with every weight at least 1/(2N), section 1.5

The constrained rules are quadratic programmes. Each is solved exactly with
the Lawson-Hanson non-negative least squares algorithm (scipy.optimize.nnls)
after a change of variables, so there is no solver setting to tune:
  minimise  x' Sigma x / 2 - mu' x   subject to x >= 0
is, with Sigma = L L' (Cholesky), the same as minimising |L' x - L^-1 mu|^2,
a non-negative least squares problem. A budget constraint 1'w = 1 is added
as one heavily weighted extra row, then the result is renormalised; a lower
bound w >= a is a shift of variables. tests/ checks every one against an
independent solver.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import nnls

from . import constants as C


def sample_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Sample mean and covariance (divisor M - 1) of an (M, N) window."""
    R = np.asarray(R, dtype=float)
    mu = R.mean(axis=0)
    Sigma = np.cov(R, rowvar=False, ddof=1)
    return mu, np.atleast_2d(Sigma)


def normalise(x: np.ndarray) -> np.ndarray:
    """DGU equation (1): w = x / |1'x|."""
    s = float(np.sum(x))
    if s == 0.0:
        raise ZeroDivisionError("portfolio weights sum to zero; cannot normalise")
    return x / abs(s)


def ew(R: np.ndarray) -> np.ndarray:
    n = np.asarray(R).shape[1]
    return np.full(n, 1.0 / n)


def mv(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    x = np.linalg.solve(Sigma, mu)
    return normalise(x)


def min_variance(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    ones = np.ones(Sigma.shape[0])
    x = np.linalg.solve(Sigma, ones)
    return x / x.sum()


def vw(market_index: int):
    """The value-weighted market: weight one on the market column, zero elsewhere."""
    def rule(R: np.ndarray) -> np.ndarray:
        w = np.zeros(np.asarray(R).shape[1])
        w[market_index] = 1.0
        return w
    rule.__name__ = "vw"
    return rule


def in_sample_sharpe(R: np.ndarray, rule=mv) -> float:
    """DGU equation (13): the Sharpe ratio of a rule estimated once on the whole sample."""
    mu, Sigma = sample_moments(R)
    w = rule(R)
    return float(w @ mu / np.sqrt(w @ Sigma @ w))


# ---------------------------------------------------------------------------
# Bayes-Stein, DGU section 1.3.2, equations (4) and (5); Jorion (1986)
# ---------------------------------------------------------------------------

def bayes_stein_from_moments(mu: np.ndarray, Sigma_mnm2: np.ndarray, M: int) -> tuple[np.ndarray, np.ndarray, float]:
    """Jorion's predictive moments from a window's mean and its covariance with divisor M - N - 2.

    Returns (mu_bs, Sigma_bs, phi). The mean is shrunk toward the mean return
    of the minimum-variance portfolio by phi (DGU equation 5); the covariance
    is inflated for the uncertainty in the mean, Jorion's predictive variance.
    """
    N = len(mu)
    Sigma_inv_1 = np.linalg.solve(Sigma_mnm2, np.ones(N))
    w_min = Sigma_inv_1 / Sigma_inv_1.sum()
    mu_min = float(mu @ w_min)
    d = mu - mu_min
    dist = float(d @ np.linalg.solve(Sigma_mnm2, d))     # (mu - mu_min 1)' Sigma^-1 (mu - mu_min 1)
    lam = (N + 2) / dist
    phi = lam / (M + lam)                                 # equals (N+2) / ((N+2) + M * dist), DGU equation (5)
    mu_bs = (1.0 - phi) * mu + phi * mu_min
    Sigma_bs = Sigma_mnm2 * (1.0 + 1.0 / (M + lam)) + (lam / (M * (M + 1.0 + lam))) * np.outer(np.ones(N), np.ones(N)) / Sigma_inv_1.sum()
    return mu_bs, Sigma_bs, phi


def bayes_stein_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """The predictive mean and covariance of Jorion (1986), as DGU implement them, from a window of returns.

    The sample covariance here divides by M - N - 2 (DGU equation 5), the
    unbiased estimator of the inverse. Returns (mu_bs, Sigma_bs, phi).
    """
    R = np.asarray(R, dtype=float)
    M, N = R.shape
    mu = R.mean(axis=0)
    dev = R - mu
    Sigma = dev.T @ dev / (M - N - 2)
    return bayes_stein_from_moments(mu, Sigma, M)


def bs(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


# ---------------------------------------------------------------------------
# Constrained rules, DGU section 1.5, solved as non-negative least squares
# ---------------------------------------------------------------------------

_BUDGET_WEIGHT = 1e6   # weight of the budget row; the residual on 1'w = 1 is then below 1e-12 and the result is renormalised anyway


def _cholesky_upper(Sigma: np.ndarray) -> np.ndarray:
    """L' with Sigma = L L'. A tiny ridge keeps a near-singular window solvable."""
    N = Sigma.shape[0]
    try:
        L = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(Sigma + 1e-12 * np.trace(Sigma) / N * np.eye(N))
    return L.T


def nonneg_mean_variance(mu: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """argmax mu'x - x'Sigma x / 2 over x >= 0, as NNLS: minimise |L'x - L^-1 mu|^2.

    This is DGU's Lagrangian (8) read literally: the non-negativity constraint
    alone, the scale then removed by normalise(). It is kept so that the two
    readings of the equation can be compared (notebook 03); budget_mean_variance
    is the reading that reproduces the paper's tables.
    """
    Lt = _cholesky_upper(Sigma)
    b = np.linalg.solve(Lt.T, mu)           # L^-1 mu
    x, _ = nnls(Lt, b)
    return x


def budget_mean_variance(mu: np.ndarray, Sigma: np.ndarray, gamma: float = C.DGU_RISK_AVERSION) -> np.ndarray:
    """argmax mu'w - gamma w'Sigma w / 2 over w >= 0 and 1'w = 1.

    The budget constraint sits inside the optimisation, so gamma no longer
    cancels: with gamma = 1 and monthly moments the risk term is small against
    the differences in means, and the rule often puts all wealth in the asset
    with the highest estimated mean. That is the "corner solutions with all
    wealth invested in a single asset" of DGU footnote 22, and it is what
    reproduces their Tables 3 to 5 (constants.DGU_CONSTRAINED_BUDGET_INSIDE).
    As NNLS: minimise |sqrt(gamma) L'w - L^-1 mu / sqrt(gamma)|^2 with the
    budget as a heavily weighted extra row.
    """
    N = len(mu)
    Lt = _cholesky_upper(Sigma)
    A = np.vstack([np.sqrt(gamma) * Lt, _BUDGET_WEIGHT * np.ones(N)])
    b = np.concatenate([np.linalg.solve(Lt.T, mu) / np.sqrt(gamma), [_BUDGET_WEIGHT]])
    w, _ = nnls(A, b)
    return w / w.sum()


def constrained_min_variance(Sigma: np.ndarray, lower: float = 0.0) -> np.ndarray:
    """argmin w'Sigma w over w >= lower, 1'w = 1, as NNLS on v = w - lower.

    Objective |L'(v + lower 1)|^2, so the NNLS target is -L' lower 1; the budget
    1'v = 1 - N lower is appended as a heavily weighted row.
    """
    N = Sigma.shape[0]
    Lt = _cholesky_upper(Sigma)
    ones = np.ones(N)
    A = np.vstack([Lt, _BUDGET_WEIGHT * ones])
    b = np.concatenate([-Lt @ (lower * ones), [_BUDGET_WEIGHT * (1.0 - N * lower)]])
    v, _ = nnls(A, b)
    w = v + lower
    return w / w.sum()


def mv_c(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def mv_c_cone(R: np.ndarray) -> np.ndarray:
    """Equation (8) read literally, for the comparison in notebook 03; not a reported rule."""
    mu, Sigma = sample_moments(R)
    return normalise(nonneg_mean_variance(mu, Sigma))


def min_c(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c(R: np.ndarray) -> np.ndarray:
    """DGU p. 1926: minimum variance with w >= a 1, a = 1/(2N), halfway between min-c and 1/N."""
    _, Sigma = sample_moments(R)
    N = Sigma.shape[0]
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / N)


# ---------------------------------------------------------------------------
# The same rules from a window's moments, for the long simulated histories of
# notebook 04, where recomputing the covariance at every month would dominate
# the running time. Each takes (mu, Sigma, M) with Sigma the sample covariance
# with divisor M - 1, and returns the same weights as its window-based twin
# (tests/test_rules_backtest.py checks the equality).
# ---------------------------------------------------------------------------

def _to_mnm2(Sigma: np.ndarray, M: int) -> np.ndarray:
    """Rescale a divisor-(M-1) covariance to divisor M - N - 2, DGU equation (5)."""
    N = Sigma.shape[0]
    return Sigma * (M - 1) / (M - N - 2)


def ew_m(mu, Sigma, M):
    return np.full(len(mu), 1.0 / len(mu))


def mv_m(mu, Sigma, M):
    return normalise(np.linalg.solve(Sigma, mu))


def min_m(mu, Sigma, M):
    x = np.linalg.solve(Sigma, np.ones(len(mu)))
    return x / x.sum()


def bs_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


def mv_c_m(mu, Sigma, M):
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / len(mu))


MOMENT_RULES = {
    "ew": ew_m,
    "mv": mv_m,
    "bs": bs_m,
    "min": min_m,
    "mv-c": mv_c_m,
    "bs-c": bs_c_m,
    "min-c": min_c_m,
    "g-min-c": g_min_c_m,
}


# The registry. vw needs the market column and is built per dataset with vw(index).
RULES = {
    "ew": ew,
    "mv": mv,
    "bs": bs,
    "min": min_variance,
    "mv-c": mv_c,
    "bs-c": bs_c,
    "min-c": min_c,
    "g-min-c": g_min_c,
}

In [ ]:
%%writefile src/bp/covariance.py
"""Four ways to estimate a covariance matrix from a window of excess returns, and the tests that judge them (notebook 10).

Every estimator takes a window of excess returns (rows are periods, columns are
assets) and returns an N by N covariance matrix for the period that follows. The
sample covariance uses all N(N+1)/2 numbers the window offers. Ledoit-Wolf
shrinkage pulls it towards a simple target. The factor model and the
principal-component model describe it with a small number of common sources of movement
plus each asset's own variance. The tests measure how well each estimate
forecasts the variance of test portfolios in the next month (the bias
statistic) and how well the minimum-variance portfolio built from it does
(the tests of Dom, Howard, Jansen and Lohre, 2024).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import rules as S

PERIODS_PER_YEAR = 12


# ---------------------------------------------------------------------------
# Estimators
# ---------------------------------------------------------------------------

def sample_cov(X: np.ndarray) -> np.ndarray:
    """The sample covariance, denominator T - 1."""
    return np.cov(np.asarray(X, dtype=float), rowvar=False, ddof=1)


def ewma_cov(X: np.ndarray, halflife: float) -> np.ndarray:
    """Exponentially weighted covariance: the weight of an observation halves every `halflife` periods, the most recent weighing most.

    The mean is weighted the same way; weights are normalised to sum to one.
    """
    X = np.asarray(X, dtype=float)
    T = X.shape[0]
    lam = 0.5 ** (1.0 / halflife)
    w = lam ** np.arange(T - 1, -1, -1)
    w = w / w.sum()
    mu = w @ X
    D = X - mu
    return (D * w[:, None]).T @ D / (1.0 - (w ** 2).sum())   # the unbiased normalisation for weighted samples


def lw_scaled_identity(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004b): (1 - k) S + k m I, with S the sample covariance (denominator T), m the average
    sample variance and k the intensity that minimises the expected squared error.

    Returns the estimate and k. The intensity is b^2 / d^2 clipped to [0, 1], where d^2 is the squared
    distance between S and m I and b^2 measures how much S moves from one observation to the next.
    """
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    m = np.trace(S) / N
    d2 = ((S - m * np.eye(N)) ** 2).sum() / N
    b2_bar = sum(((np.outer(x, x) - S) ** 2).sum() for x in D) / (N * T * T)
    b2 = min(b2_bar, d2)
    k = float(b2 / d2) if d2 > 0 else 0.0
    return (1.0 - k) * S + k * m * np.eye(N), k


def lw_constant_correlation(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004a): shrink S towards the matrix with the same variances and one common correlation,
    the average sample correlation; the intensity follows the paper's appendix. Returns the estimate and k."""
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    var = np.diag(S)
    sd = np.sqrt(var)
    corr = S / np.outer(sd, sd)
    rbar = (corr.sum() - N) / (N * (N - 1))
    F = rbar * np.outer(sd, sd)
    np.fill_diagonal(F, var)
    # pi: the sum over i, j of the estimated asymptotic variances of the entries of sqrt(T) S
    Y = D ** 2
    pi_mat = (Y.T @ Y) / T - S ** 2
    pi = pi_mat.sum()
    # rho: the diagonal part plus the covariance between the entries of S and the target's correlation structure
    term1 = ((D ** 3).T @ D) / T          # theta_{ii,ij}: average over t of x_it^3 x_jt
    help_ = term1 - var[:, None] * S      # (1/T) sum_t (x_it^2 - S_ii)(x_it x_jt - S_ij)
    theta_ii = help_
    theta_jj = help_.T
    rho_off = (rbar / 2.0) * (np.sqrt(var[None, :] / var[:, None]) * theta_ii + np.sqrt(var[:, None] / var[None, :]) * theta_jj)
    np.fill_diagonal(rho_off, 0.0)
    rho = np.trace(pi_mat) + rho_off.sum()
    gamma = ((F - S) ** 2).sum()
    k = (pi - rho) / gamma / T if gamma > 0 else 0.0
    k = float(min(1.0, max(0.0, k)))
    return (1.0 - k) * S + k * F, k


def factor_cov(X: np.ndarray, F: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """The exact factor model: X regressed on the factors F (with an intercept) over the window;
    the estimate is B Sigma_F B' plus the diagonal of residual variances. Returns the estimate and B (N by K)."""
    X = np.asarray(X, dtype=float)
    F = np.asarray(F, dtype=float)
    T, K = F.shape
    A = np.column_stack([np.ones(T), F])
    coef, *_ = np.linalg.lstsq(A, X, rcond=None)      # (K + 1) by N
    B = coef[1:].T                                     # N by K
    resid = X - A @ coef
    Sigma_F = np.cov(F, rowvar=False, ddof=1).reshape(K, K)
    resid_var = resid.var(axis=0, ddof=K + 1)
    return B @ Sigma_F @ B.T + np.diag(resid_var), B


def pca_cov(X: np.ndarray, n_components: int, market: np.ndarray | None = None) -> tuple[np.ndarray, int]:
    """The principal-component model.

    With `market` given (constants.PCA_FIRST_COMPONENT == "market"): each asset is regressed on the market
    series, and the covariance of the residuals is described by its first n_components - 1 principal
    components plus the diagonal of what they leave; the estimate is beta beta' var(market) plus that.
    Without a market series, the first n_components of X's own covariance plus the diagonal remainder.
    Returns the estimate and the number of components taken from the data.
    """
    X = np.asarray(X, dtype=float)
    if market is not None:
        m = np.asarray(market, dtype=float)
        A = np.column_stack([np.ones(len(m)), m])
        coef, *_ = np.linalg.lstsq(A, X, rcond=None)
        beta = coef[1]
        resid = X - A @ coef
        base = np.outer(beta, beta) * m.var(ddof=1)
        k = n_components - 1
        target = np.cov(resid, rowvar=False, ddof=2)
    else:
        base = 0.0
        k = n_components
        target = sample_cov(X)
    if k <= 0:
        return base + np.diag(np.diag(target)), 0
    vals, vecs = np.linalg.eigh(target)
    order = np.argsort(vals)[::-1][:k]
    V = vecs[:, order]
    L = np.clip(vals[order], 0.0, None)
    common = (V * L) @ V.T
    own = np.diag(np.clip(np.diag(target) - np.diag(common), 1e-12, None))
    return base + common + own, k


def scale_daily_to_monthly(Sigma_daily: np.ndarray, scale: float = C.DAILY_TO_MONTHLY_SCALE) -> np.ndarray:
    """A daily covariance times the number of trading days in a month; exact when daily returns do not predict the next day's."""
    return Sigma_daily * scale


def variance_ratio(x: np.ndarray, q: int = C.COV_SCALE_VR_HORIZON_DAYS) -> float:
    """The variance ratio of Lo and MacKinlay (1988): the variance of the overlapping sums of q consecutive
    values divided by q times the variance of the single values. It is 1 when the values do not predict each
    other, above 1 when a high value tends to be followed by a high one (positive autocorrelation) and below 1
    when a high value tends to be followed by a low one. Multiplying a daily covariance by q times this ratio,
    in place of q alone, gives the covariance of q-day sums that the autocorrelation implies."""
    v = np.asarray(x, dtype=float)
    if len(v) < 2 * q:
        return float("nan")
    sums = np.convolve(v, np.ones(q), mode="valid")
    return float(sums.var(ddof=1) / (q * v.var(ddof=1)))


# ---------------------------------------------------------------------------
# Windows
# ---------------------------------------------------------------------------

def monthly_window(excess_m: pd.DataFrame, month: pd.Period, months: int = C.EXT_ESTIMATION_WINDOW) -> pd.DataFrame:
    """The `months` monthly rows before `month`."""
    end = month - 1
    start = month - months
    return excess_m.loc[start:end]


def daily_window(excess_d: pd.DataFrame, month: pd.Period, years: int) -> pd.DataFrame:
    """The trading days of the 12 * years calendar months before `month`, days with any missing value left out."""
    end = (month - 1).asfreq("D", "end")
    start = (month - 12 * years).asfreq("D", "start")
    return excess_d.loc[start:end].dropna()


def daily_benchmark_excess(returns_d: pd.DataFrame, rf_d: pd.Series, weights_m: pd.DataFrame) -> pd.Series:
    """The benchmark's daily excess return: each day's industry returns weighted by that month's benchmark weights, minus the daily risk-free rate."""
    month_of_day = returns_d.index.asfreq("M")
    w = weights_m.reindex(month_of_day)
    w.index = returns_d.index
    return ((w * returns_d).sum(axis=1, min_count=returns_d.shape[1]) - rf_d.reindex(returns_d.index)).rename("benchmark_excess")


# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

def bias_statistic(realised: np.ndarray, predicted_var: np.ndarray) -> float:
    """The standard deviation of realised / sqrt(predicted variance): 1 when the forecasts are calibrated, above 1 when risk is under-forecast."""
    z = np.asarray(realised, dtype=float) / np.sqrt(np.asarray(predicted_var, dtype=float))
    return float(z.std(ddof=1))


def random_active_weights(n_assets: int, draws: int = C.COV_TEST_ACTIVE_DRAWS, bound: float = C.ACTIVE_WEIGHT_BOUND,
                          seed: int = C.COV_TEST_SEED) -> np.ndarray:
    """`draws` active-weight vectors: weights sum to zero and the largest absolute weight equals `bound`."""
    rng = np.random.default_rng(seed)
    U = rng.uniform(-1.0, 1.0, size=(draws, n_assets))
    U = U - U.mean(axis=1, keepdims=True)
    return bound * U / np.abs(U).max(axis=1, keepdims=True)


def min_variance_unconstrained(Sigma: np.ndarray) -> np.ndarray:
    """w proportional to Sigma^-1 1, scaled to sum to one; negative weights allowed."""
    x = np.linalg.solve(Sigma, np.ones(Sigma.shape[0]))
    return x / x.sum()


def min_variance_long_only(Sigma: np.ndarray) -> np.ndarray:
    """The long-only minimum-variance portfolio, solved exactly by non-negative least squares (rules.constrained_min_variance)."""
    return S.constrained_min_variance(Sigma, 0.0)


def condition_number(Sigma: np.ndarray) -> float:
    vals = np.linalg.eigvalsh(Sigma)
    return float(vals.max() / max(vals.min(), 1e-300))


def annualised_vol(x: pd.Series | np.ndarray) -> float:
    return float(np.std(np.asarray(x, dtype=float), ddof=1) * np.sqrt(PERIODS_PER_YEAR))


# ---------------------------------------------------------------------------
# The panels and the estimator variants, as notebooks 10 to 12 use them
# ---------------------------------------------------------------------------

class Panels:
    """Everything the estimators read: monthly and daily excess returns of the 49 industries, the six factors at both
    frequencies, the benchmark's weights and its excess return at both frequencies, and the plain monthly returns."""

    def __init__(self, start: str = C.EXT_SAMPLE_START):
        from . import french_loader as fl
        from . import benchmark as BM
        inputs = BM.load_inputs()
        f3 = fl.load_monthly("factors3", 0)
        f5 = fl.load_monthly("factors5", 0)
        mom = fl.load_monthly("momentum", 0)
        start_p = pd.Period(start, "M")
        self.returns_m = inputs["returns"].loc[start_p:]
        self.last = self.returns_m.index[-1]
        self.rf_m = f3["RF"].loc[start_p:self.last]
        self.excess_m = self.returns_m.sub(self.rf_m, axis=0)
        self.weights = BM.cap_weights(inputs["firms"].loc[start_p:], inputs["size"].loc[start_p:], C.CAPW_WEIGHT_TIMING)
        self.bench_m = (BM.combination_return(self.returns_m, self.weights) - self.rf_m).rename("benchmark_excess")
        self.factors_m = pd.concat([f5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], mom["Mom"]], axis=1).loc[start_p:self.last]
        returns_d = fl.load_monthly("ind49_daily", r"Value Weighted Returns -- Daily").loc[start_p.asfreq("D", "start"):]
        f3d = fl.load_monthly("factors3_daily", 0)
        f5d = fl.load_monthly("factors5_daily", 0)
        momd = fl.load_monthly("momentum_daily", 0)
        self.rf_d = f3d["RF"].reindex(returns_d.index)
        self.returns_d = returns_d
        self.excess_d = returns_d.sub(self.rf_d, axis=0)
        self.factors_d = pd.concat([f5d[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], momd["Mom"]], axis=1).reindex(returns_d.index)
        self.bench_d = daily_benchmark_excess(returns_d, self.rf_d, self.weights)
        self.files = ("ind49", "factors3", "factors5", "momentum", "ind49_daily", "factors3_daily", "factors5_daily", "momentum_daily")


DAILY_YEARS = {"daily_3y": C.EXT_COV_DAILY_WINDOW_YEARS, "daily_1y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[0],
               "daily_5y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1], "daily_5y_ewma": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1]}


def estimate(name: str, freq: str, month: pd.Period, P: Panels) -> tuple[np.ndarray, float, int]:
    """One estimator variant for `month`, built on the data before it: the monthly covariance (daily ones scaled by
    DAILY_TO_MONTHLY_SCALE), the shrinkage intensity (nan for the others) and the number of observations in the window.
    The same dispatch as notebook 10's evaluation loop."""
    if freq == "monthly_120":
        X = monthly_window(P.excess_m, month); F = P.factors_m.loc[X.index]; mk = P.bench_m.loc[X.index]; scale = 1.0
    else:
        X = daily_window(P.excess_d, month, DAILY_YEARS[freq]); F = P.factors_d.loc[X.index]; mk = P.bench_d.loc[X.index]; scale = C.DAILY_TO_MONTHLY_SCALE
    Xv = X.to_numpy(); intensity = np.nan
    if name == "sample":
        S = ewma_cov(Xv, C.EXT_COV_EWMA_HALFLIFE_DAYS) if freq.endswith("ewma") else sample_cov(Xv)
    elif name == "ledoit_wolf":
        S, intensity = lw_scaled_identity(Xv)
    elif name == "ledoit_wolf_cc":
        S, intensity = lw_constant_correlation(Xv)
    elif name == "factor":
        S, _ = factor_cov(Xv, F.to_numpy())
    elif name == "pca":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS, market=mk.to_numpy())
    elif name == "pca_2":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS_SENSITIVITY, market=mk.to_numpy())
    else:
        raise KeyError(name)
    return S * scale, intensity, len(X)


# The five estimator variants of the rolling evaluation (notebook 12): the four estimators on three years of daily
# returns and the paper's convention, the sample covariance of 120 monthly returns.
EVALUATION_VARIANTS = tuple((e, "daily_3y") for e in C.COV_ESTIMATORS) + (("sample", "monthly_120"),)
# The robust portfolio (constants.ROBUST_VARIANT) is built from all five covariances at once, and RiskMetrics
# (constants.EWMA_VARIANT) is the recency-weighted sample covariance; the seven together are the variants the rolling evaluation reports.
ALL_VARIANTS = EVALUATION_VARIANTS + (C.ROBUST_VARIANT, C.EWMA_VARIANT)

In [ ]:
%%writefile src/bp/optimiser.py
"""The optimiser of the extension: the portfolio that tracks the benchmark most closely under a set of constraints (notebook 11).

For a month with benchmark weights b and a monthly covariance matrix Sigma, the
optimiser finds the weights w that minimise the forecast tracking error, the
square root of 12 (w - b)' Sigma (w - b), subject to the constraints of a named
set: the weights sum to one; no short positions (long_only); each untilted
industry within ACTIVE_WEIGHT_BOUND of its benchmark weight (active_weight_bound);
one-way turnover from the drifted previous portfolio at most
TURNOVER_CAP_MONTHLY_ONE_WAY (turnover_cap); the tilted industries at no more than
TILT_MAX_SHARE_OF_BENCHMARK of their benchmark weight (tilt); named industries
at zero (exclusion); the portfolio's beta to the benchmark equal to the benchmark's
(beta_neutral). The objective is quadratic and every constraint is linear,
so the problem is a convex quadratic program with one global minimum, solved by
cvxpy with the solver named in constants.OPT_SOLVER. Two safeguards (notebook 11,
check D5): when the turnover cap is among the constraints, the smallest feasible
one-way turnover is found first, by a linear program, so that the solver is never
handed a capped problem that has no solution (a cap below that turnover is raised
to it and the month is marked relaxed); and every solution is checked against its
constraints within constants.OPT_TOL before it is returned, whatever status the
solver reports.
"""
from __future__ import annotations

import warnings
from dataclasses import dataclass, field

import cvxpy as cp
import numpy as np

from . import constants as C

PERIODS_PER_YEAR = 12


class OptimiserError(RuntimeError):
    """The solver did not return an optimal solution, or the solution failed a check."""


@dataclass
class Solution:
    weights: np.ndarray
    active: np.ndarray                 # weights minus benchmark weights
    tracking_error: float              # forecast, per year: sqrt(12 a' Sigma a)
    turnover: float                    # one-way, against the previous portfolio (nan when there is none)
    turnover_cap_used: float | None    # the cap applied, after any relaxation
    relaxed: bool                      # True when the cap had to be raised to make the month feasible
    status: str
    binding: dict = field(default_factory=dict)   # constraint name -> count of bounds within OPT_TOL of their limit


def cholesky_factor(Sigma: np.ndarray) -> np.ndarray:
    """L with Sigma = L L', after symmetrising; a jitter of 1e-10 on the diagonal is added only if Sigma is not positive definite."""
    S = 0.5 * (Sigma + Sigma.T)
    try:
        return np.linalg.cholesky(S)
    except np.linalg.LinAlgError:
        return np.linalg.cholesky(S + 1e-10 * np.eye(S.shape[0]))


def drifted_weights(w_prev: np.ndarray, returns_prev: np.ndarray) -> np.ndarray:
    """The previous portfolio after the month's returns: w_i (1 + r_i) divided by the portfolio's own gross return."""
    grown = np.asarray(w_prev, dtype=float) * (1.0 + np.asarray(returns_prev, dtype=float))
    return grown / grown.sum()


def forecast_tracking_error(w: np.ndarray, b: np.ndarray, Sigma: np.ndarray) -> float:
    """sqrt(12 a' Sigma a) with a = w - b, in return units per year."""
    a = np.asarray(w, dtype=float) - np.asarray(b, dtype=float)
    return float(np.sqrt(max(PERIODS_PER_YEAR * a @ Sigma @ a, 0.0)))


def effective_number(w: np.ndarray) -> float:
    """1 / sum of squared weights: 49 for 1/N over 49 industries, smaller for a concentrated portfolio."""
    w = np.asarray(w, dtype=float)
    return float(1.0 / (w @ w))


def one_way_turnover(w: np.ndarray, w0: np.ndarray) -> float:
    """Half the sum of absolute weight changes: the fraction of the portfolio sold (and bought) at the rebalance."""
    return float(0.5 * np.abs(np.asarray(w, dtype=float) - np.asarray(w0, dtype=float)).sum())


def benchmark_betas(Sigma: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Each asset's beta to the benchmark under Sigma: Sigma b divided by b' Sigma b; the benchmark's own beta is one."""
    Sb = np.asarray(Sigma, dtype=float) @ np.asarray(b, dtype=float)
    return Sb / float(np.asarray(b, dtype=float) @ Sb)


def active_variance_parts(a: np.ndarray, Sigma: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    """Split a' Sigma a into the market part, (a' beta)^2 times the benchmark's variance, and the rest.

    With beta = Sigma b / (b' Sigma b) the two parts add up to a' Sigma a exactly: the market part is the
    variance of the active return's projection on the benchmark, the rest is the variance of what the
    benchmark leaves unexplained."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float); Sigma = np.asarray(Sigma, dtype=float)
    beta = benchmark_betas(Sigma, b)
    var_b = float(b @ Sigma @ b)
    market = float((a @ beta) ** 2 * var_b)
    total = float(a @ Sigma @ a)
    return market, total - market


def bounds(b: np.ndarray, constraints: tuple, tilt_mask: np.ndarray, exclude_mask: np.ndarray,
           active_bound: float = C.ACTIVE_WEIGHT_BOUND, tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK) -> tuple[np.ndarray, np.ndarray]:
    """The lower and upper bound on each weight that the named constraints imply."""
    N = len(b)
    lower = np.full(N, -np.inf)
    upper = np.full(N, np.inf)
    if "long_only" in constraints:
        lower = np.maximum(lower, 0.0)
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        lower = np.where(untilted, np.maximum(lower, b - active_bound), lower)
        upper = np.where(untilted, np.minimum(upper, b + active_bound), upper)
    if "tilt" in constraints:
        lower = np.where(tilt_mask, np.maximum(lower, 0.0), lower)
        upper = np.where(tilt_mask, np.minimum(upper, tilt_share * b), upper)
    if "exclusion" in constraints:
        lower = np.where(exclude_mask, 0.0, lower)
        upper = np.where(exclude_mask, 0.0, upper)
    return lower, upper


def _build(Ls: list, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None,
           objective: str = "tracking_error", betas: list | None = None):
    """The cvxpy problem. `Ls` holds one Cholesky factor for the single-covariance problem and one per covariance for the robust
    one, whose objective is the largest of the forecast active variances; `betas` holds the beta vectors the
    beta-neutral constraint applies to, one per covariance."""
    N = len(b)
    w = cp.Variable(N)
    cons = [cp.sum(w) == 1]
    finite_lo = np.isfinite(lower)
    finite_up = np.isfinite(upper)
    if finite_lo.any():
        cons.append(w[finite_lo] >= lower[finite_lo])
    if finite_up.any():
        cons.append(w[finite_up] <= upper[finite_up])
    for beta in betas or []:
        cons.append(beta @ (w - b) == 0)
    turnover_expr = None
    if w0 is not None:
        # one-way turnover through explicit purchases p and sales n, w - w0 = p - n: the same quantity as half the sum of
        # absolute changes, in the form in which trades are written as buys and sells. The solver's status is not relied on
        # in either form (notebook 11, check D5: on a month with no feasible portfolio the solver reported "optimal" with
        # weights that fail the budget); check() tests every returned portfolio, and _solve poses a capped problem only
        # after the smallest feasible turnover is known.
        p = cp.Variable(N, nonneg=True)
        n = cp.Variable(N, nonneg=True)
        cons.append(w - w0 == p - n)
        turnover_expr = 0.5 * cp.sum(p + n)
        if cap is not None:
            cons.append(turnover_expr <= cap)
    if objective == "tracking_error" and len(Ls) == 1:
        obj = cp.Minimize(cp.sum_squares(Ls[0].T @ (w - b)))
    elif objective == "tracking_error":                     # robust: the largest active standard deviation across the covariances
        t = cp.Variable()                                   # written with norms (second-order cones) and in percent, which the solver handles more accurately than squares
        cons += [cp.norm(100.0 * (L.T @ (w - b)), 2) <= t for L in Ls]
        obj = cp.Minimize(t)
    else:                                                   # the smallest feasible one-way turnover
        obj = cp.Minimize(turnover_expr)
    return w, cp.Problem(obj, cons)


def solve(Sigma: np.ndarray, b: np.ndarray, constraints: tuple, w_prev: np.ndarray | None = None,
          tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
          turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
          tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> Solution:
    """The constrained tracking-error minimiser for one month.

    `constraints` is a tuple of names from constants.CONSTRAINT_SETS. `w_prev` is the previous portfolio after
    drift; it is required when "turnover_cap" is among the constraints. If the cap makes the month infeasible,
    the cap is raised to the smallest feasible one-way turnover and the solution says so.
    """
    return _solve([np.asarray(Sigma, dtype=float)], b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)


def solve_robust(Sigmas: list, b: np.ndarray, constraints: tuple, w_prev: np.ndarray | None = None,
                 tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
                 turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
                 tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> Solution:
    """The robust minimiser: the weights whose largest forecast active variance across the covariances in `Sigmas`
    is smallest, under the same constraints; beta neutrality, when asked, holds under the average of the covariances'
    beta vectors. The tracking error reported is the worst case across the covariances."""
    return _solve([np.asarray(S, dtype=float) for S in Sigmas], b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)


def _solve(Sigmas: list, b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol) -> Solution:
    b = np.asarray(b, dtype=float)
    N = len(b)
    tilt_mask = np.zeros(N, bool) if tilt_mask is None else np.asarray(tilt_mask, bool)
    exclude_mask = np.zeros(N, bool) if exclude_mask is None else np.asarray(exclude_mask, bool)
    if "turnover_cap" in constraints and w_prev is None:
        raise OptimiserError("the turnover cap needs the previous portfolio")
    if "exclusion" in constraints and (exclude_mask is None or not np.asarray(exclude_mask, bool).any()):
        raise OptimiserError("the exclusion constraint needs a mask naming at least one industry")
    lower, upper = bounds(b, constraints, tilt_mask, exclude_mask, active_bound, tilt_share)
    if (lower > upper + tol).any():
        raise OptimiserError("a weight's lower bound exceeds its upper bound; the constraint set is contradictory for this month")
    Ls = [cholesky_factor(S) for S in Sigmas]
    w0 = None if w_prev is None else np.asarray(w_prev, dtype=float)
    cap = turnover_cap if "turnover_cap" in constraints else None
    # beta neutrality: one equality. With one covariance, its own betas; with more than one (the robust problem), the average
    # of their beta vectors, because one equality per covariance made the 1979-07 problem infeasible under the mandate's
    # bounds, and the minimax objective already penalises the market exposure that remains under each covariance.
    betas = [np.mean([benchmark_betas(S, b) for S in Sigmas], axis=0)] if "beta_neutral" in constraints else None
    relaxed = False

    options = C.OPT_SOLVER_OPTIONS if len(Ls) == 1 else C.OPT_SOLVER_OPTIONS_ROBUST
    accepted = ("optimal",) if len(Ls) == 1 else ("optimal", "optimal_inaccurate")
    if cap is not None:
        # feasibility first: the smallest one-way turnover the other constraints allow, from a linear program. A cap below
        # it would hand the solver a problem with no solution, and on such a problem the solver's status is not reliable
        # (notebook 11, check D5); so the cap is raised to that turnover, with a margin of one part in a million plus tol,
        # and the month is marked relaxed, before the capped problem is posed.
        w_t, prob_t = _build(Ls, b, lower, upper, w0, None, objective="turnover", betas=betas)
        _run(prob_t, solver, options)
        if prob_t.status not in ("optimal", "optimal_inaccurate"):
            raise OptimiserError(f"infeasible without the turnover cap: {prob_t.status}")
        check(_clean(w_t.value), b, lower, upper, w0, None, tol)      # the program's own weights pass the checks too
        if float(prob_t.value) > cap + 10 * tol:
            cap = float(prob_t.value) * (1.0 + 1e-6) + tol
            relaxed = True
    w, prob = _build(Ls, b, lower, upper, w0, cap, betas=betas)
    _run(prob, solver, options)
    x = _clean(w.value)
    if prob.status not in accepted:
        raise OptimiserError(f"solver status {prob.status} on a problem known to be feasible")
    check(x, b, lower, upper, w0, cap, tol)
    a = x - b
    binding = {}
    if "long_only" in constraints:
        binding["long_only"] = int(((x <= tol) & ~exclude_mask).sum())
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        binding["active_weight_bound"] = int((untilted & (np.abs(np.abs(a) - active_bound) <= tol)).sum())
    if "tilt" in constraints:
        binding["tilt"] = int((tilt_mask & (np.abs(x - tilt_share * b) <= tol)).sum())
    to = one_way_turnover(x, w0) if w0 is not None else float("nan")
    if "turnover_cap" in constraints:
        binding["turnover_cap"] = int(abs(to - cap) <= 10 * tol)
    for beta in betas or []:
        if abs(beta @ a) > 10 * tol:
            raise OptimiserError(f"the portfolio's beta differs from the benchmark's by {beta @ a:.2e}")
    te = max(forecast_tracking_error(x, b, S) for S in Sigmas)      # the worst case across the covariances; one covariance, its own
    return Solution(weights=x, active=a, tracking_error=te, turnover=to,
                    turnover_cap_used=cap, relaxed=relaxed, status=prob.status, binding=binding)


def _run(prob, solver: str, options: dict) -> None:
    """Solve with the tight stopping rule; if the solver reports the result as inaccurate at that rule, solve again at
    its default rule (about 1e-8). A single-covariance problem is accepted only when reported optimal; the robust
    problem, written with second-order cones, is accepted as inaccurate too once the feasibility checks pass, and
    the status is recorded so that the evaluation can count such months."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Solution may be inaccurate")   # that case is solved again below at the default rule
        prob.solve(solver=solver, **options)
        if prob.status == "optimal_inaccurate":
            prob.solve(solver=solver)


def _clean(value) -> np.ndarray:
    if value is None:
        return np.full(0, np.nan)
    x = np.asarray(value, dtype=float).ravel()
    return np.where(np.abs(x) < 1e-12, 0.0, x)


def feasible(w: np.ndarray, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None, tol: float) -> bool:
    """True when the weights sum to one, lie inside their bounds and respect the turnover cap, all within tol."""
    if len(w) != len(b) or not np.isfinite(w).all() or abs(w.sum() - 1.0) > tol:
        return False
    if (w < lower - tol).any() or (w > upper + tol).any():
        return False
    if cap is not None and w0 is not None and one_way_turnover(w, w0) > cap + 10 * tol:
        return False
    return True


def check(w: np.ndarray, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None, tol: float) -> None:
    """Raise unless the weights sum to one, lie inside their bounds and respect the turnover cap, all within tol."""
    if len(w) != len(b) or not np.isfinite(w).all():
        raise OptimiserError("the solver returned no weights")
    if abs(w.sum() - 1.0) > tol:
        raise OptimiserError(f"weights sum to {w.sum():.8f}")
    if (w < lower - tol).any() or (w > upper + tol).any():
        raise OptimiserError("a weight lies outside its bounds")
    if cap is not None and w0 is not None and one_way_turnover(w, w0) > cap + 10 * tol:
        raise OptimiserError(f"turnover {one_way_turnover(w, w0):.6f} exceeds the cap {cap:.6f}")


def masks(columns, tilt=C.TILT_INDUSTRIES, exclude=()) -> tuple[np.ndarray, np.ndarray]:
    """Boolean masks over `columns` for the tilted and the excluded industries; a name that is not a column raises."""
    cols = [str(c).strip() for c in columns]
    for name in tuple(tilt) + tuple(exclude):
        if name not in cols:
            raise KeyError(f"{name} is not one of the {len(cols)} industries")
    return np.array([c in tilt for c in cols]), np.array([c in exclude for c in cols])


# ---------------------------------------------------------------------------
# Version 2, module A: the factor objective under the mandate (notebook 14)
# ---------------------------------------------------------------------------
# A second problem beside the first, which stays as it is. For a month with
# benchmark weights b, a monthly covariance Sigma, the industries' betas B to
# the six factors (N by K, estimated before the month as in notebook 13) and
# the target vector s (constants.FO_TARGET_SIGN), the optimiser finds the
# weights w that maximise the targeted exposure s' B' (w - b) under every
# constraint of the named set and the budget: the forecast tracking error
# sqrt(12 (w - b)' Sigma (w - b)) at most tau. The objective is linear and the
# budget a second-order cone, so the problem is a convex program with one
# global maximum, solved by cvxpy with the same solver as the first problem.
# In a month in which the budget cannot be met (the tracking-error-minimising
# portfolio of the first problem already exceeds tau), the path takes that
# portfolio, the month is marked and counted, and nothing else is relaxed
# (constants.FO_INFEASIBLE_RULE). Every returned portfolio is checked against
# its bounds, the turnover cap and the budget, whatever the solver's status.

@dataclass
class FactorSolution(Solution):
    budget: float = float("nan")              # tau, per year
    budget_met: bool = True                   # False in a month that took the tracking-error-minimising portfolio
    objective: float = float("nan")           # s' B' (w - b), the targeted exposure the month bought
    exposures: np.ndarray = field(default_factory=lambda: np.full(0, np.nan))   # B' (w - b), one per factor


def targeted_exposure(B: np.ndarray, s: np.ndarray, w: np.ndarray, b: np.ndarray) -> float:
    """s' B' (w - b): the sum of the active exposures to the targeted factors, each weighted by its sign in s."""
    return float(np.asarray(s, dtype=float) @ np.asarray(B, dtype=float).T @ (np.asarray(w, dtype=float) - np.asarray(b, dtype=float)))


def _build_factor(Ls: list, b: np.ndarray, g: np.ndarray, tau_m: float, lower: np.ndarray, upper: np.ndarray,
                  w0: np.ndarray | None, cap: float | None, betas: list | None):
    """The cvxpy problem of the factor objective: maximise g' (w - b) with g = B s, under the bounds, the beta
    equalities, the turnover cap written as buys and sells, and one budget cone per covariance factor in `Ls`
    (one for the single-covariance problem, five for the robust one). Written in percent, as the robust problem is."""
    N = len(b)
    w = cp.Variable(N)
    cons = [cp.sum(w) == 1]
    finite_lo = np.isfinite(lower)
    finite_up = np.isfinite(upper)
    if finite_lo.any():
        cons.append(w[finite_lo] >= lower[finite_lo])
    if finite_up.any():
        cons.append(w[finite_up] <= upper[finite_up])
    for beta in betas or []:
        cons.append(beta @ (w - b) == 0)
    if w0 is not None:
        p = cp.Variable(N, nonneg=True)
        n = cp.Variable(N, nonneg=True)
        cons.append(w - w0 == p - n)
        if cap is not None:
            cons.append(0.5 * cp.sum(p + n) <= cap)
    cons += [cp.norm(100.0 * (L.T @ (w - b)), 2) <= 100.0 * tau_m for L in Ls]
    return w, cp.Problem(cp.Maximize(g @ (w - b)), cons)


def solve_factor_objective(Sigma, b: np.ndarray, B: np.ndarray, s: np.ndarray, budget_annual: float, constraints: tuple,
                           w_prev: np.ndarray | None = None, tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
                           turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
                           tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> FactorSolution:
    """The factor objective under the mandate for one month.

    `Sigma` is one monthly covariance, or a list of them for the robust version, in which the budget holds under
    every covariance at once and beta neutrality under the average of their beta vectors. `B` is N by K, `s` the
    K signs, `budget_annual` tau per year. If the turnover cap alone makes the month infeasible it is raised to the
    smallest feasible turnover, as in the first problem. If the budget cannot be met, the month takes the
    tracking-error-minimising portfolio of the first problem and `budget_met` is False.
    """
    Sigmas = [np.asarray(S, dtype=float) for S in (Sigma if isinstance(Sigma, (list, tuple)) else [Sigma])]
    b = np.asarray(b, dtype=float)
    B = np.asarray(B, dtype=float)
    s = np.asarray(s, dtype=float)
    N = len(b)
    if B.shape[0] != N or B.shape[1] != len(s):
        raise OptimiserError(f"the betas are {B.shape} for {N} industries and {len(s)} targets")
    if not budget_annual > 0:
        raise OptimiserError("the budget must be positive")
    tilt_mask = np.zeros(N, bool) if tilt_mask is None else np.asarray(tilt_mask, bool)
    exclude_mask = np.zeros(N, bool) if exclude_mask is None else np.asarray(exclude_mask, bool)
    if "turnover_cap" in constraints and w_prev is None:
        raise OptimiserError("the turnover cap needs the previous portfolio")
    if "exclusion" in constraints and not exclude_mask.any():
        raise OptimiserError("the exclusion constraint needs a mask naming at least one industry")
    lower, upper = bounds(b, constraints, tilt_mask, exclude_mask, active_bound, tilt_share)
    if (lower > upper + tol).any():
        raise OptimiserError("a weight's lower bound exceeds its upper bound; the constraint set is contradictory for this month")
    Ls = [cholesky_factor(S) for S in Sigmas]
    w0 = None if w_prev is None else np.asarray(w_prev, dtype=float)
    cap = turnover_cap if "turnover_cap" in constraints else None
    betas = [np.mean([benchmark_betas(S, b) for S in Sigmas], axis=0)] if "beta_neutral" in constraints else None
    g = B @ s
    tau_m = float(budget_annual) / np.sqrt(PERIODS_PER_YEAR)
    relaxed = False
    options = C.OPT_SOLVER_OPTIONS_ROBUST            # the budget is a cone, so the robust problem's stopping rule applies
    if cap is not None:
        # feasibility first, as in the first problem: the smallest one-way turnover the other constraints allow
        w_t, prob_t = _build(Ls, b, lower, upper, w0, None, objective="turnover", betas=betas)
        _run(prob_t, solver, options)
        if prob_t.status not in ("optimal", "optimal_inaccurate"):
            raise OptimiserError(f"infeasible without the turnover cap: {prob_t.status}")
        check(_clean(w_t.value), b, lower, upper, w0, None, tol)
        if float(prob_t.value) > cap + 10 * tol:
            cap = float(prob_t.value) * (1.0 + 1e-6) + tol
            relaxed = True
    w, prob = _build_factor(Ls, b, g, tau_m, lower, upper, w0, cap, betas)
    _run(prob, solver, options)
    x = _clean(w.value)
    met = prob.status in ("optimal", "optimal_inaccurate") and len(x) == N and np.isfinite(x).all()
    if met:
        met = feasible(x, b, lower, upper, w0, cap, tol) and max(forecast_tracking_error(x, b, S) for S in Sigmas) <= budget_annual * (1.0 + 1e-6) + 10 * tol
    if not met:
        # the budget cannot be met this month: the tracking-error-minimising portfolio of the first problem, under the same
        # constraints and the same cap, is the month's portfolio; its forecast must indeed exceed the budget
        base = _solve(Sigmas, b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)
        if base.tracking_error <= budget_annual * (1.0 - 1e-6) - 10 * tol:
            raise OptimiserError(f"the solver reported {prob.status} on a budget of {budget_annual:.4f} that the minimiser meets at {base.tracking_error:.4f}")
        x, cap, relaxed, status = base.weights, base.turnover_cap_used, base.relaxed, base.status
    else:
        status = prob.status
    check(x, b, lower, upper, w0, cap, tol)
    a = x - b
    for beta in betas or []:
        if abs(beta @ a) > 10 * tol:
            raise OptimiserError(f"the portfolio's beta differs from the benchmark's by {beta @ a:.2e}")
    binding = {}
    if "long_only" in constraints:
        binding["long_only"] = int(((x <= tol) & ~exclude_mask).sum())
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        binding["active_weight_bound"] = int((untilted & (np.abs(np.abs(a) - active_bound) <= tol)).sum())
    if "tilt" in constraints:
        binding["tilt"] = int((tilt_mask & (np.abs(x - tilt_share * b) <= tol)).sum())
    to = one_way_turnover(x, w0) if w0 is not None else float("nan")
    if "turnover_cap" in constraints:
        binding["turnover_cap"] = int(abs(to - cap) <= 10 * tol)
    te = max(forecast_tracking_error(x, b, S) for S in Sigmas)
    binding["budget"] = int(met and abs(te - budget_annual) <= C.FO_E1_BIND_TOL_ANNUAL)
    return FactorSolution(weights=x, active=a, tracking_error=te, turnover=to, turnover_cap_used=cap, relaxed=relaxed, status=status,
                          binding=binding, budget=float(budget_annual), budget_met=bool(met), objective=float(g @ a), exposures=B.T @ a)

In [ ]:
%%writefile src/bp/evaluation.py
"""The rolling evaluation of the extension (notebook 12): one portfolio path per specification, month by month.

A path is one variant (a covariance estimator, or the robust portfolio built from all of them) under one
constraint set. It starts as the index. In every evaluation month the covariance is estimated on the data before
the month, the optimiser solves for the weights with the drifted previous weights as the turnover reference, the
portfolio and the benchmark earn the month's returns, and the active return, the forecast tracking error, the
turnover and the state of the cap are recorded. The measures of a path are computed from its records: realised
tracking error, forecast tracking error and its calibration, active return gross and net of trading costs,
turnover and relaxed-cap months, holdings.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import optimiser as O

PERIODS_PER_YEAR = 12


def run_path(months, Sigmas, weights: pd.DataFrame, returns: pd.DataFrame, constraints: tuple,
             tilt_mask: np.ndarray, exclude_mask: np.ndarray | None, robust: bool = False, keep_weights: bool = False) -> pd.DataFrame:
    """One path. `Sigmas` maps a month to its covariance (or to the list of covariances when `robust`).

    Returns one record per month: forecast tracking error (per year), active return, one-way turnover, the cap
    used and whether it was relaxed, the effective number of holdings, the industries held, and the weights when
    asked. The previous portfolio of the first month is the benchmark of that month (constants.EVAL_START_PORTFOLIO).
    """
    records = []
    w_prev = None
    prev_month = None
    for month in months:
        b = weights.loc[month].to_numpy()
        r = returns.loc[month].to_numpy()
        if w_prev is None:
            w0 = b.copy()
            b_drift = b.copy()
        else:
            w0 = O.drifted_weights(w_prev, returns.loc[prev_month].to_numpy())
            b_drift = O.drifted_weights(weights.loc[prev_month].to_numpy(), returns.loc[prev_month].to_numpy())
        kwargs = dict(w_prev=w0, tilt_mask=tilt_mask, exclude_mask=exclude_mask)
        sol = O.solve_robust(Sigmas[month], b, constraints, **kwargs) if robust else O.solve(Sigmas[month], b, constraints, **kwargs)
        rec = {"month": month, "forecast_te": sol.tracking_error, "active_return": float(sol.weights @ r - b @ r),
               "turnover": sol.turnover, "cap_used": sol.turnover_cap_used if "turnover_cap" in constraints else np.nan,
               "relaxed": bool(sol.relaxed), "effective_number": O.effective_number(sol.weights), "held": int((sol.weights > 1e-6).sum()),
               "status": sol.status,
               "benchmark_turnover": O.one_way_turnover(b, b_drift)}       # the index's own turnover: how far its weights moved beyond what its returns did
        if keep_weights:
            rec["weights"] = sol.weights
        records.append(rec)
        w_prev, prev_month = sol.weights, month
    return pd.DataFrame(records).set_index("month")


def realised_tracking_error(active: pd.Series | np.ndarray) -> float:
    """The standard deviation of the monthly active returns, times the square root of 12."""
    return float(np.std(np.asarray(active, dtype=float), ddof=1) * np.sqrt(PERIODS_PER_YEAR))


def bias_statistic(active: np.ndarray, forecast_te: np.ndarray, scale: np.ndarray | None = None) -> float:
    """sd(active return / forecast monthly standard deviation); `scale` multiplies the forecast variance (the variance-ratio
    correction). nan when a forecast is zero, as under the sets in which the portfolio is the benchmark."""
    f = np.asarray(forecast_te, dtype=float) / np.sqrt(PERIODS_PER_YEAR)
    if scale is not None:
        f = f * np.sqrt(np.asarray(scale, dtype=float))
    if (np.asarray(forecast_te, dtype=float) <= C.OPT_TOL_TE).any():     # a forecast of zero (the portfolio is the benchmark) has no calibration
        return float("nan")
    return float(np.std(np.asarray(active, dtype=float) / f, ddof=1))


def summarise(path: pd.DataFrame, vr_scale: pd.Series | None = None) -> dict:
    """The measures of one path, in return units per year where they are returns or risks."""
    active = path["active_return"].to_numpy()
    te = realised_tracking_error(active)
    gross = PERIODS_PER_YEAR * float(np.mean(active))
    turnover = float(path["turnover"].mean())
    out = {"realised TE": te, "forecast TE (mean)": float(path["forecast_te"].mean()),
           "bias": bias_statistic(active, path["forecast_te"].to_numpy()),
           "bias (variance ratio)": bias_statistic(active, path["forecast_te"].to_numpy(), None if vr_scale is None else vr_scale.loc[path.index].to_numpy()),
           "active return (gross)": gross, "turnover (one-way)": turnover, "benchmark turnover": float(path["benchmark_turnover"].mean()),
           "relaxed months": int(path["relaxed"].sum()), "relaxed share": float(path["relaxed"].mean()),
           "inaccurate months": int((path["status"] != "optimal").sum()),
           "effective number": float(path["effective_number"].mean()), "industries held": float(path["held"].mean())}
    for c in C.EXT_COST_LEVELS:
        bp = int(round(c * 1e4))
        cost = PERIODS_PER_YEAR * turnover * c
        out[f"cost at {bp} bp"] = cost
        out[f"net active return at {bp} bp"] = gross - cost
        out[f"information ratio at {bp} bp"] = (gross - cost) / te if te > C.OPT_TOL_TE else float("nan")   # no ratio for a portfolio that is the benchmark
    return out


def by_decade(path: pd.DataFrame, first_label: str, last_label: str) -> pd.Series:
    """Realised tracking error by decade of the evaluation month; the first and last partial decades carry the given labels."""
    def label(m):
        if m.year < 1990:
            return first_label
        if m.year >= 2020:
            return last_label
        return f"{(m.year // 10) * 10}s"
    groups = pd.Series([label(m) for m in path.index], index=path.index)
    return path.groupby(groups)["active_return"].apply(realised_tracking_error)


def calibrated_forecast(path: pd.DataFrame, window: int = C.EVAL_CALIBRATION_WINDOW, minimum: int = C.EVAL_CALIBRATION_MIN) -> pd.Series:
    """The forecast tracking error of month t multiplied by the ratio of realised to forecast tracking error over the
    previous `window` months of the same path (square root of the mean squared active return over the mean forecast
    variance), using only months before t; unchanged while fewer than `minimum` months are available."""
    f_var = (path["forecast_te"] / np.sqrt(PERIODS_PER_YEAR)) ** 2
    a2 = path["active_return"] ** 2
    num = a2.shift(1).rolling(window, min_periods=minimum).mean()
    den = f_var.shift(1).rolling(window, min_periods=minimum).mean()
    ratio = np.sqrt(num / den)
    ratio = ratio.where(den > 0).fillna(1.0)
    return path["forecast_te"] * ratio

## Count first: the benchmark weights, the drifted weights, the tilted industries, the specification count

In [ ]:
import sys
sys.path.insert(0, "src")
import time
import numpy as np
import pandas as pd
import cvxpy
from bp import constants as C
from bp import french_loader as fl
from bp import covariance as CV
from bp import optimiser as O

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)

P = CV.Panels()
weights, returns_m = P.weights, P.returns_m
eval_months = pd.period_range(C.COV_EVAL_START, P.last, freq="M")
industries = [c.strip() for c in weights.columns]
tilt_mask, exclude_mask = O.masks(weights.columns)

# Expectation 1
assert np.allclose(weights.sum(axis=1), 1.0, atol=1e-9), "benchmark weights do not sum to one in every month"
drift = pd.DataFrame({m: O.drifted_weights(weights.loc[m - 1].to_numpy(), returns_m.loc[m - 1].to_numpy()) for m in weights.index[1:]}, index=weights.columns).T
drift.index = pd.PeriodIndex(drift.index, freq="M")
assert len(drift) == len(weights) - 1 and np.allclose(drift.sum(axis=1), 1.0, atol=1e-9)
assert tilt_mask.sum() == len(C.TILT_INDUSTRIES) == 3
spec_count = len(CV.ALL_VARIANTS) * len(C.CONSTRAINT_SETS)
assert spec_count == C.EXT_SPEC_COUNT == 28
provenance = [fl.download(key) for key in P.files]
fl.write_provenance(provenance)
vintage = provenance[0]["vintage_line"].split("the ")[1].split(" ")[0]
print(f"{len(weights)} months of benchmark weights, {weights.index[0]} to {weights.index[-1]}, each summing to one; drifted weights for {len(drift)} months, each summing to one")
print(f"tilted industries: {', '.join(C.TILT_INDUSTRIES)}, held at no more than {C.TILT_MAX_SHARE_OF_BENCHMARK:.0%} of their benchmark weight; at {weights.index[-1]} they are "
      + ", ".join(f"{n} {100 * weights.loc[weights.index[-1], c]:.2f}%" for n, c in zip(C.TILT_INDUSTRIES, weights.columns[tilt_mask])) + " of the benchmark")
print(f"specification count: {spec_count} ({len(CV.ALL_VARIANTS)} variants, five estimators, the robust portfolio and RiskMetrics, times {len(C.CONSTRAINT_SETS)} constraint sets); cvxpy {cvxpy.__version__}, solver {C.OPT_SOLVER}")
print(f"expectation 1 holds; CRSP vintage of the eight files: {vintage}")

## The optimiser at three months

This is the first part of the notebook: C3 is the tilt alone, the first of its two versions. The report months are the first evaluation month (1979-07, the first with both a 120-month and a three-year daily window), the paper's last month (2004-11) and the vintage's last month. At each, the five estimator variants of the rolling evaluation (the four estimators on three years of daily returns and the sample covariance of 120 monthly returns) give five covariance matrices, and the optimiser runs under each constraint set with the benchmark as the previous portfolio. The table reports, for each portfolio, the forecast tracking error in basis points a year, the one-way turnover, the cap after any relaxation, how many industries it holds, its effective number of holdings, the active weights of the three tilted industries in percentage points, and how many constraints of each kind bind.

In [ ]:
SETS = {"C0": C.CONSTRAINT_SETS["C0"], "C1": C.CONSTRAINT_SETS["C1"], "C2": C.CONSTRAINT_SETS["C2"], "C3": C.CONSTRAINT_SET_TILT_ONLY}   # the first run: C3 as designed on 10 September, the tilt alone
report_months = [eval_months[0] if m == "first_eval" else (P.last if m == "last" else pd.Period(m, "M")) for m in C.OPT_REPORT_MONTHS]
tilt_cols = list(weights.columns[tilt_mask])

Sigmas = {}
t0 = time.time()
for month in report_months:
    for v in CV.EVALUATION_VARIANTS:
        Sigmas[(month, v)], _, _ = CV.estimate(*v, month, P)
print(f"{len(Sigmas)} covariance matrices built in {time.time() - t0:.0f}s")

rows, solutions = [], {}
for month in report_months:
    b = weights.loc[month].to_numpy()
    for v in CV.EVALUATION_VARIANTS:
        S = Sigmas[(month, v)]
        for cs, names in SETS.items():
            sol = O.solve(S, b, names, w_prev=b, tilt_mask=tilt_mask)
            solutions[(month, v, cs)] = sol
            row = {"month": str(month), "estimator": v[0], "data": v[1], "set": cs,
                   "tracking error (bp a year)": 1e4 * sol.tracking_error, "one-way turnover": sol.turnover,
                   "cap used": sol.turnover_cap_used if "turnover_cap" in names else np.nan, "cap relaxed": sol.relaxed,
                   "industries held": int((sol.weights > 1e-6).sum()), "effective number": O.effective_number(sol.weights)}
            for name, j in zip(C.TILT_INDUSTRIES, np.flatnonzero(tilt_mask)):
                row[f"active {name} (pp)"] = 100 * sol.active[j]
            for k, n in sol.binding.items():
                row[f"binding: {k}"] = n
            rows.append(row)
report = pd.DataFrame(rows).set_index(["month", "estimator", "data", "set"])
print(report.to_string(float_format=lambda x: f"{x:.2f}"))

## Expectations 2 and 3: the known answers and the cost of the tilt

In [ ]:
te = report["tracking error (bp a year)"] / 1e4
pairs = [(m, v) for m in report_months for v in CV.EVALUATION_VARIANTS]
def TE(m, v, cs):
    return te.loc[(str(m), v[0], v[1], cs)]

e2a = sum(TE(m, v, "C0") < C.OPT_TOL_TE and TE(m, v, "C1") < C.OPT_TOL_TE for m, v in pairs)
e2b = sum(report.loc[(str(m), v[0], v[1], "C3"), "binding: tilt"] == 3 and TE(m, v, "C3") > C.OPT_TOL_TE for m, v in pairs)
e2c_mono = sum(TE(m, v, "C1") >= TE(m, v, "C0") - C.OPT_TOL_TE and TE(m, v, "C2") >= TE(m, v, "C1") - C.OPT_TOL_TE for m, v in pairs)
e2c_tilt = sum(TE(m, v, "C3") > TE(m, v, "C2") for m, v in pairs)
n = len(pairs)
print(f"expectation 2a: under C0 and C1 the solution is the benchmark (forecast tracking error below {C.OPT_TOL_TE:g} a year) in {e2a} of {n} month-estimator pairs: {'MET' if e2a == n else 'NOT MET'}")
print(f"expectation 2b: under C3 all three tilts bind and the tracking error is positive in {e2b} of {n}: {'MET' if e2b == n else 'NOT MET'}")
print(f"expectation 2c: the tracking error does not fall from C0 to C1 to C2 in {e2c_mono} of {n} ({'MET' if e2c_mono == n else 'NOT MET'}), and rises from C2 to C3 in {e2c_tilt} of {n} ({'MET' if e2c_tilt == n else 'NOT MET'})")
last = report_months[-1]
te_tilt = TE(last, ("sample", "daily_3y"), "C3")
print(f"expectation 3: at {last} the forecast tracking error under C3 with the sample covariance on daily data is {1e4 * te_tilt:.1f} basis points a year "
      f"(level {1e4 * C.OPT_TILT_TE_MAX_ANNUAL:.0f}): {'MET' if te_tilt < C.OPT_TILT_TE_MAX_ANNUAL else 'NOT MET'}")
print()
c3 = report.xs("C3", level="set")["tracking error (bp a year)"].unstack(["estimator", "data"])
print("the cost of the tilt by month and estimator, forecast tracking error under C3 in basis points a year:")
print(c3.to_string(float_format=lambda x: f"{x:.1f}"))

## Expectation 4: where the tilt's weight goes, the effective number of holdings, which constraints bind

The tilt takes weight out of Coal, Oil and Utilities, and the optimiser puts it where the forecast tracking error grows least. Two things decide where that is. First, the portfolio's beta to the benchmark: Oil and Utilities are low-beta industries (their returns move less than one for one with the market), so removing them leaves a portfolio with a beta above one, and the cheapest way to bring it back to one is to buy other low-beta industries. Second, among the low-beta industries, those whose remaining movement (after the market is taken out) is correlated with Oil and Utilities, and whose own volatility is low, replace the removed exposure at the least added risk. The table lists, at the last month under C3 with the sample covariance on daily data, the industries with the largest positive active weights beside each one's beta to the benchmark, its own volatility, and its correlation with Oil and with Utilities after the market is taken out; the rank correlations below the table say which of these numbers predicts the active weights. One more thing shapes the list: under C3 the first month's cap is relaxed to the smallest feasible turnover, so the portfolio trades exactly what the tilt requires and nothing else, and the purchases concentrate in the few industries with the best hedge per unit traded.

In [ ]:
b_last = weights.loc[last].to_numpy()
S_last = Sigmas[(last, ("sample", "daily_3y"))]
sol = solutions[(last, ("sample", "daily_3y"), "C3")]
own_vol = np.sqrt(12 * np.diag(S_last))
beta_bench = (S_last @ b_last) / (b_last @ S_last @ b_last)
resid = S_last - np.outer(S_last @ b_last, S_last @ b_last) / (b_last @ S_last @ b_last)     # covariance after the benchmark is taken out
resid_corr = resid / np.sqrt(np.outer(np.diag(resid), np.diag(resid)))
i_oil, i_util = industries.index("Oil"), industries.index("Util")
where = pd.DataFrame({"benchmark weight %": 100 * b_last, "active weight (pp)": 100 * sol.active, "beta to benchmark": beta_bench, "own volatility %": 100 * own_vol,
                      "residual corr with Oil": resid_corr[:, i_oil], "residual corr with Util": resid_corr[:, i_util]}, index=industries)
print(f"where the tilt's weight goes at {last} (C3, sample covariance on daily data); the tilt removes {-100 * sol.active[tilt_mask].sum():.2f} percentage points from Coal, Oil and Util, "
      f"whose betas to the benchmark are {', '.join(f'{beta_bench[j]:.2f}' for j in np.flatnonzero(tilt_mask))}:")
print(where.sort_values("active weight (pp)", ascending=False).head(10).to_string(float_format=lambda x: f"{x:.2f}"))
print()
print(f"{int((sol.active[~tilt_mask] > 1e-6).sum())} of the 46 untilted industries receive weight; the portfolio's beta to the benchmark is {float(sol.weights @ beta_bench):.3f}")
sp = where.loc[~tilt_mask].corr(method="spearman")["active weight (pp)"]
print("rank correlation across the 46 untilted industries between the active weight and: " + "; ".join(f"{c} {sp[c]:.2f}" for c in ["beta to benchmark", "own volatility %", "residual corr with Oil", "residual corr with Util", "benchmark weight %"]))
print()
eff = report["effective number"].unstack("set")
eff.insert(0, "benchmark", [O.effective_number(weights.loc[pd.Period(m, 'M')].to_numpy()) for m in eff.index.get_level_values("month")])
print("effective number of holdings (49 would be 1/N):")
print(eff.to_string(float_format=lambda x: f"{x:.1f}"))
print()
bind_cols = [c for c in report.columns if c.startswith("binding")]
print("constraints binding (count of bounds at their limit), by set, averaged over the fifteen month-estimator pairs:")
print(report[bind_cols].groupby(level="set").mean().to_string(float_format=lambda x: f"{x:.1f}"))

## Expectation 4, continued: the tilt's cost with and without the transition cap

Reported, no verdict. Under C3 at a report month the previous portfolio is the benchmark, so the cap is relaxed to the smallest feasible turnover and the portfolio trades exactly the tilt and nothing else. The same tilt with free trading (long-only, active bound and tilt, no cap) is the cost of the tilt once the portfolio has settled, which notebook 12's rolling evaluation reaches after the first months. The two are reported side by side.

In [ ]:
# Added after the first run (4 October 2026): the tilt under the minimum-turnover transition (C3) against the tilt with free trading.
rows = []
for month in report_months:
    b = weights.loc[month].to_numpy()
    for v in CV.EVALUATION_VARIANTS:
        s3 = solutions[(month, v, "C3")]
        sf = O.solve(Sigmas[(month, v)], b, ("long_only", "active_weight_bound", "tilt"), w_prev=b, tilt_mask=tilt_mask)
        rows.append({"month": str(month), "estimator": v[0], "data": v[1], "C3, transition (bp)": 1e4 * s3.tracking_error, "turnover C3": s3.turnover,
                     "tilt, free trading (bp)": 1e4 * sf.tracking_error, "turnover free": sf.turnover, "industries held, free": int((sf.weights > 1e-6).sum())})
free = pd.DataFrame(rows).set_index(["month", "estimator", "data"])
print(free.to_string(float_format=lambda x: f"{x:.2f}"))

## Expectation 5: the turnover cap as a speed limit

A stress case at the last month: the fund starts as 1/N, far from the cap-weighted benchmark, and may trade 2% one way a month. Under C1 it reaches the benchmark at once, because nothing limits the trade. Under long-only plus the cap it moves towards the benchmark 2% a month, choosing each month the trade that lowers the forecast tracking error most; the benchmark weights and the covariance are held fixed so that only the cap acts. Under C2 the active bound is hard and 1/N is far outside it, so the cap is relaxed to the smallest turnover that reaches the bound.

In [ ]:
start = np.ones(49) / 49
distance = O.one_way_turnover(start, b_last)
sol_c1 = O.solve(S_last, b_last, SETS["C1"], w_prev=start, tilt_mask=tilt_mask)
path, w_now = [], start.copy()
for step in range(1, max(C.OPT_SPEED_LIMIT_STEPS) + 1):
    s = O.solve(S_last, b_last, ("long_only", "turnover_cap"), w_prev=w_now, tilt_mask=tilt_mask)
    path.append({"rebalance": step, "one-way turnover": s.turnover, "forecast tracking error (bp a year)": 1e4 * s.tracking_error,
                 "one-way distance to the benchmark": O.one_way_turnover(s.weights, b_last)})
    w_now = s.weights
path = pd.DataFrame(path).set_index("rebalance")
first = path.loc[1]
e5 = abs(first["one-way turnover"] - C.TURNOVER_CAP_MONTHLY_ONE_WAY) <= C.OPT_TOL and first["forecast tracking error (bp a year)"] > 0 and sol_c1.tracking_error < C.OPT_TOL_TE
print(f"from 1/N at {last}: one-way distance to the benchmark {distance:.3f}, which is {distance / C.TURNOVER_CAP_MONTHLY_ONE_WAY:.0f} months of trading at the cap if every trade reduced it")
print(f"expectation 5: under long-only plus the cap the first rebalance trades {first['one-way turnover']:.4f} one way (cap {C.TURNOVER_CAP_MONTHLY_ONE_WAY}) and leaves a forecast tracking error of {first['forecast tracking error (bp a year)']:.0f} basis points; "
      f"under C1 the first rebalance reaches the benchmark (tracking error {1e4 * sol_c1.tracking_error:.4f} basis points, turnover {sol_c1.turnover:.3f}): {'MET' if e5 else 'NOT MET'}")
print()
print("the journey from 1/N under long-only plus the cap, benchmark and covariance held fixed:")
print(path.loc[list(C.OPT_SPEED_LIMIT_STEPS)].to_string(float_format=lambda x: f"{x:.4f}"))
sol_c2 = O.solve(S_last, b_last, SETS["C2"], w_prev=start, tilt_mask=tilt_mask)
print()
print(f"under C2 from 1/N the active bound demands a one-way turnover of {sol_c2.turnover:.3f} at once, so the cap is relaxed to it ({'relaxed' if sol_c2.relaxed else 'not relaxed'}); forecast tracking error {1e4 * sol_c2.tracking_error:.1f} basis points, the cost of landing inside the bound in one month")

## Part two: the practical problem behind the hedge (D1 to D4)

A tilt removes exposure, and the optimiser replaces it with whatever the covariance says is closest. Four things about that replacement matter to a manager, each checked here with its expectation fixed in `constants.py` before the code (4 October 2026):

- **D1, how much of the tilt is a market bet.** The forecast active variance a' Sigma a splits exactly into a market part, (a' beta)^2 times the benchmark's variance, with beta the industries' betas to the benchmark under the same covariance, and a residual part, the variance of what the benchmark leaves unexplained. Expectation D1a: the market part is below 5% of the whole in all fifteen month-estimator pairs, because the market is the largest direction of the covariance, so a beta away from one is the most expensive deviation and the optimiser removes it on its own.
- **D2, forcing the beta to one.** A beta-neutral constraint, the portfolio's beta equal to the benchmark's, is added to C3 as a new switchable constraint. Expectation D2a: it raises the forecast tracking error by less than 1 basis point a year in every pair, for the same reason. The lesson, if it holds: with a tracking-error objective and a usable covariance, a manager does not need to impose beta neutrality by hand; the optimiser's first move is to restore it.
- **D3, is the hedge consistent with the mandate.** At 2026-08 the tobacco industry (Smoke) received weight as a low-beta substitute for Utilities. A sustainability mandate that underweights carbon would not buy tobacco to hedge it. Reported, no verdict: the forecast tracking error of C3 plus the exclusion of Smoke, against C3, at every report month and estimator; the expectation written before the code is a rise below 2 basis points, because the other low-beta industries are close substitutes.
- **D4, how much the hedge depends on the covariance and on the month.** (a) The rank correlation between the C3 active-weight vectors of each pair of estimators at the same month, over the 46 untilted industries; expectation D4a: at every report month the four daily estimators agree with each other more than any of them agrees with the monthly sample covariance. (b) The cross-evaluation matrix at the last month: the C3 portfolio built with estimator i (the row), its tracking error forecast with estimator j's covariance (the column). Within a column the diagonal is the minimum by construction, because the column's estimator built that portfolio to minimise its own forecast and the other rows' portfolios are feasible under the same set; so the report is the excess of each entry over its column's diagonal, in basis points: how much worse the column's estimator thinks another estimator's portfolio is than its own. (c) The three industries that absorb most of the tilt at each report month under the sample covariance on daily data.

In [ ]:
# Added after the first run (4 October 2026): D1 and D2, the market part of the tilt and the beta-neutral constraint.
rows = []
for month in report_months:
    b = weights.loc[month].to_numpy()
    for v in CV.EVALUATION_VARIANTS:
        S = Sigmas[(month, v)]
        s3 = solutions[(month, v, "C3")]
        market, rest = O.active_variance_parts(s3.active, S, b)
        beta = O.benchmark_betas(S, b)
        s_bn = O.solve(S, b, C.CONSTRAINT_SET_TILT_ONLY + ("beta_neutral",), w_prev=b, tilt_mask=tilt_mask)
        rows.append({"month": str(month), "estimator": v[0], "data": v[1], "C3 tracking error (bp)": 1e4 * s3.tracking_error,
                     "market share of active variance": market / (market + rest), "portfolio beta under C3": float(s3.weights @ beta),
                     "C3 + beta-neutral (bp)": 1e4 * s_bn.tracking_error, "rise (bp)": 1e4 * (s_bn.tracking_error - s3.tracking_error)})
d12 = pd.DataFrame(rows).set_index(["month", "estimator", "data"])
print(d12.to_string(float_format=lambda x: f"{x:.4f}"))
d1a = int((d12["market share of active variance"] < C.OPT_D1_MARKET_SHARE_MAX).sum())
d2a = int((d12["rise (bp)"] < 1e4 * C.OPT_D2_TE_RISE_MAX).sum())
print()
print(f"D1a: the market part is below {C.OPT_D1_MARKET_SHARE_MAX:.0%} of the active variance in {d1a} of {len(d12)} pairs (largest share {d12['market share of active variance'].max():.4f}; "
      f"portfolio beta under C3 between {d12['portfolio beta under C3'].min():.3f} and {d12['portfolio beta under C3'].max():.3f}): {'MET' if d1a == len(d12) else 'NOT MET'}")
print(f"D2a: the beta-neutral constraint raises the tracking error by less than {1e4 * C.OPT_D2_TE_RISE_MAX:.0f} basis point in {d2a} of {len(d12)} pairs (largest rise {d12['rise (bp)'].max():.3f} bp): {'MET' if d2a == len(d12) else 'NOT MET'}")

In [ ]:
# Added after the first run (4 October 2026): D3, the cost of refusing the tobacco hedge, and D4, the dependence of the hedge on the estimator and the month.
_, smoke_mask = O.masks(weights.columns, exclude=C.OPT_D3_EXCLUDE)
rows = []
for month in report_months:
    b = weights.loc[month].to_numpy()
    for v in CV.EVALUATION_VARIANTS:
        s3 = solutions[(month, v, "C3")]
        s_ex = O.solve(Sigmas[(month, v)], b, C.CONSTRAINT_SET_TILT_ONLY + ("exclusion",), w_prev=b, tilt_mask=tilt_mask, exclude_mask=smoke_mask)
        rows.append({"month": str(month), "estimator": v[0], "data": v[1], "C3 (bp)": 1e4 * s3.tracking_error, "active Smoke under C3 (pp)": 100 * s3.active[smoke_mask][0],
                     "C3 without Smoke (bp)": 1e4 * s_ex.tracking_error, "rise (bp)": 1e4 * (s_ex.tracking_error - s3.tracking_error)})
d3 = pd.DataFrame(rows).set_index(["month", "estimator", "data"])
print(f"D3, the cost of excluding {', '.join(C.OPT_D3_EXCLUDE)} from the hedge:")
print(d3.to_string(float_format=lambda x: f"{x:.2f}"))
print()

labels = [f"{v[0]}, {v[1].replace('_', ' ')}" for v in CV.EVALUATION_VARIANTS]
daily = [i for i, v in enumerate(CV.EVALUATION_VARIANTS) if v[1] == "daily_3y"]
monthly = [i for i, v in enumerate(CV.EVALUATION_VARIANTS) if v[1] == "monthly_120"]
d4a_holds = []
for month in report_months:
    A_act = pd.DataFrame({labels[i]: solutions[(month, v, "C3")].active[~tilt_mask] for i, v in enumerate(CV.EVALUATION_VARIANTS)})
    rc = A_act.corr(method="spearman")
    dd = np.mean([rc.iloc[i, j] for i in daily for j in daily if i < j])
    dm = np.mean([rc.iloc[i, j] for i in daily for j in monthly])
    d4a_holds.append(dd > dm)
    print(f"D4a at {month}: rank correlation of the C3 active weights across estimators (46 untilted industries); daily-daily mean {dd:.2f}, daily-monthly mean {dm:.2f}")
    print(rc.to_string(float_format=lambda x: f"{x:.2f}"))
    print()
print(f"D4a: the daily estimators agree with each other more than with the monthly sample covariance at {sum(d4a_holds)} of {len(report_months)} report months: {'MET' if all(d4a_holds) else 'NOT MET'}")
print()
cross = pd.DataFrame(index=labels, columns=labels, dtype=float)
for i, vi in enumerate(CV.EVALUATION_VARIANTS):
    w_i = solutions[(last, vi, "C3")].weights
    for j, vj in enumerate(CV.EVALUATION_VARIANTS):
        cross.iloc[i, j] = 1e4 * O.forecast_tracking_error(w_i, b_last, Sigmas[(last, vj)])
excess = cross.sub(np.diag(cross.to_numpy()), axis=1)
assert (excess.to_numpy() >= -1e-6).all(), "a column's own portfolio must be that column's minimum"
print(f"D4b at {last}: forecast tracking error (bp a year) of the C3 portfolio built with the row's estimator, judged by the column's covariance:")
print(cross.to_string(float_format=lambda x: f"{x:.1f}"))
print()
print("excess over the column's own portfolio (bp): how much worse the column's estimator thinks the row's portfolio is than its own")
print(excess.to_string(float_format=lambda x: f"{x:.1f}"))
print()
print("D4c: the three industries that absorb most of the tilt, sample covariance on daily data, by report month:")
for month in report_months:
    a = pd.Series(solutions[(month, ("sample", "daily_3y"), "C3")].active, index=industries)
    top = a.sort_values(ascending=False).head(3)
    print(f"  {month}: " + ", ".join(f"{k} +{100 * x:.2f} pp" for k, x in top.items()) + f"; tilt removes {-100 * a[tilt_mask].sum():.2f} pp")

## The fixes (F1 to F3): the mandate's C3 and the robust portfolio at the three months

D1 to D4 describe problems a manager would fix, so the design was changed (4 October 2026, before notebook 12's code), with the expectations written into `constants.py` first:

- **Fix 1 (D1, D2).** Beta neutrality joins C3 for every covariance. It cost the daily estimators at most 1.2 basis points and removes the beta bet the monthly covariance left standing. Factor neutrality beyond the market is not imposed: the market is 55% of the industries' variance (notebook 09) and the next components 17%, and factor betas from a window carry their own estimation error.
- **Fix 2 (D3).** The optimiser receives the whole mandate: the carbon tilt and the exclusions of tobacco (Smoke) and weapons (Guns), the standard exclusions of Dutch institutional mandates, both of which the tilt's hedge had bought. Excluded industries are at weight zero, which costs tracking error of its own (together 0.9% of the 2026-08 benchmark).
- **Fix 3 (D4).** A robust portfolio for model risk: the weights whose largest forecast tracking error across the five covariances is smallest, so that no single matrix is trusted; one more convex problem, with beta neutrality under every covariance. It joins notebook 12's rolling evaluation as a sixth variant.

Expectations: F1, under the mandate's C3 the portfolio's beta equals one within 1e-6 and Smoke and Guns have weight zero, in all fifteen pairs. F2, the mandate's C3 costs more than the tilt alone in every pair (its feasible set is smaller) and the rise is below 15 basis points a year everywhere; D2 and D3 together suggest up to about 12 for the monthly covariance at 2004-11. F3, the robust portfolio's worst forecast tracking error across the five covariances is at most the worst of every single-estimator portfolio (a check, it is the minimax by construction), and under each covariance its excess over that covariance's own minimum is at most the largest excess any single-estimator portfolio shows there (F3a, a statement about the data: the compromise is nobody's worst case). Reported: the robust portfolio's mean excess and where its weight goes.

In [ ]:
# The fixes (4 October 2026): the mandate's C3 and the robust portfolio at the three report months.
_, mandate_mask = O.masks(weights.columns, exclude=C.MANDATE_EXCLUSIONS)
fix_rows, fixed, robust = [], {}, {}
for month in report_months:
    b = weights.loc[month].to_numpy()
    for v in CV.EVALUATION_VARIANTS:
        S = Sigmas[(month, v)]
        s_old = solutions[(month, v, "C3")]
        s_new = O.solve(S, b, C.CONSTRAINT_SETS["C3"], w_prev=b, tilt_mask=tilt_mask, exclude_mask=mandate_mask)
        fixed[(month, v)] = s_new
        beta = O.benchmark_betas(S, b)
        fix_rows.append({"month": str(month), "estimator": v[0], "data": v[1], "tilt only (bp)": 1e4 * s_old.tracking_error, "mandate C3 (bp)": 1e4 * s_new.tracking_error,
                         "rise (bp)": 1e4 * (s_new.tracking_error - s_old.tracking_error), "beta": float(s_new.weights @ beta),
                         "Smoke + Guns weight": float(s_new.weights[mandate_mask].sum()), "one-way turnover": s_new.turnover, "industries held": int((s_new.weights > 1e-6).sum())})
    robust[month] = O.solve_robust([Sigmas[(month, v)] for v in CV.EVALUATION_VARIANTS], b, C.CONSTRAINT_SETS["C3"], w_prev=b, tilt_mask=tilt_mask, exclude_mask=mandate_mask)
fixes = pd.DataFrame(fix_rows).set_index(["month", "estimator", "data"])
print("the mandate's C3 against the tilt alone:")
print(fixes.to_string(float_format=lambda x: f"{x:.4f}"))
f1 = int((((fixes["beta"] - 1).abs() <= 1e-5) & (fixes["Smoke + Guns weight"].abs() <= C.OPT_TOL)).sum())
f2 = int(((fixes["rise (bp)"] > -1e-6) & (fixes["rise (bp)"] < 1e4 * C.OPT_FIX_RISE_MAX)).sum())
print()
print(f"F1: beta equal to one and Smoke and Guns at zero in {f1} of {len(fixes)} pairs: {'MET' if f1 == len(fixes) else 'NOT MET'}")
print(f"F2: the mandate's C3 costs more than the tilt alone and less than {1e4 * C.OPT_FIX_RISE_MAX:.0f} basis points more in {f2} of {len(fixes)} pairs (largest rise {fixes['rise (bp)'].max():.1f} bp, at {fixes['rise (bp)'].idxmax()}): {'MET' if f2 == len(fixes) else 'NOT MET'}")
print()
# F3: the robust portfolio against the single-estimator portfolios, under each covariance
f3_check, f3a, rob_rows = [], [], []
for month in report_months:
    b = weights.loc[month].to_numpy()
    covs = [Sigmas[(month, v)] for v in CV.EVALUATION_VARIANTS]
    own = np.array([fixed[(month, v)].tracking_error for v in CV.EVALUATION_VARIANTS])
    te_single = np.array([[O.forecast_tracking_error(fixed[(month, vi)].weights, b, Sj) for Sj in covs] for vi in CV.EVALUATION_VARIANTS])   # row i built, column j judged
    te_rob = np.array([O.forecast_tracking_error(robust[month].weights, b, Sj) for Sj in covs])
    worst_single = te_single.max(axis=1)
    f3_check.append(bool(te_rob.max() <= worst_single.min() + 1e-7))
    excess_single = te_single - own[None, :]          # excess of portfolio i over estimator j's own minimum, under j
    excess_rob = te_rob - own
    f3a.append(bool((excess_rob <= excess_single.max(axis=0) + 1e-9).all()))
    rob_rows.append({"month": str(month), "robust worst case (bp)": 1e4 * te_rob.max(), "best single worst case (bp)": 1e4 * worst_single.min(),
                     "robust mean excess (bp)": 1e4 * excess_rob.mean(), "largest single excess (bp)": 1e4 * excess_single[~np.eye(5, dtype=bool)].max(),
                     "robust turnover": robust[month].turnover, "industries held": int((robust[month].weights > 1e-6).sum())})
rob_table = pd.DataFrame(rob_rows).set_index("month")
print("F3, the robust portfolio under the mandate's C3:")
print(rob_table.to_string(float_format=lambda x: f"{x:.2f}"))
print(f"F3 check (the robust worst case is at most every single portfolio's worst case): {sum(f3_check)} of {len(f3_check)} months; "
      f"F3a (under each covariance the robust excess is at most the largest single-portfolio excess): {sum(f3a)} of {len(f3a)} months: {'MET' if all(f3a) else 'NOT MET'}")
print()
a_rob = pd.Series(robust[last].active, index=industries)
print(f"where the robust portfolio's weight goes at {last} (mandate's C3): " + ", ".join(f"{k} +{100 * x:.2f} pp" for k, x in a_rob.sort_values(ascending=False).head(5).items())
      + f"; Smoke and Guns {100 * robust[last].weights[mandate_mask].sum():.2f}%, tilt removes {-100 * a_rob[tilt_mask].sum():.2f} pp")
a_new = pd.Series(fixed[(last, ("sample", "daily_3y"))].active, index=industries)
print(f"where the sample covariance's weight goes under the mandate's C3 at {last}: " + ", ".join(f"{k} +{100 * x:.2f} pp" for k, x in a_new.sort_values(ascending=False).head(5).items()))

## Part three: a check on the solver (D5)

A solver returns two things, a status word ("optimal", "optimal_inaccurate", "infeasible" or "unbounded") and a list of 49 weights, and both can be wrong at once. The first run of notebook 12 found such a case. At 1984-07, a July in which French rebuilds the industry portfolios from the end-of-June SIC codes, the index itself moved 4.3% one way, above the cap of 2%, so no portfolio satisfied the cap and the tilt at once; the right answer was "infeasible". The solver returned "optimal" and weights that summed to 1.0037. The independent check of the returned weights (`optimiser.check`: the weights sum to one, lie inside their bounds and respect the cap, each within 1e-6) failed them and the run stopped, which is what the check is for.

Why it matters to a manager: the optimiser's output is the trade list. A portfolio whose weights sum to 1.0037 buys 0.37% more than the fund's money, an unintended loan or a failed settlement; one that trades 22% of the fund in a month with a 2% cap pays eleven times the cost budget, 11 basis points at 50 basis points per unit of turnover against 1, and breaks the promise made to the client. A solver's status is its own opinion of its arithmetic and says nothing about the mandate. A firm answers the mandate question with a check that does not belong to the solver, the pre-trade compliance test of the order management system, which tests the proposed portfolio against every rule before an order leaves; `optimiser.check` is the same thing in miniature.

The check D5 reruns that month on the final form of the problem, with its expectation in `constants.py` first: (a) at 1984-07 under the tilt-only set, with the previous portfolio taken from the robust path run from 1979-07, the smallest one-way turnover that satisfies the tilt and the active bound exceeds the 2% cap, so the month is infeasible under the cap; (b) handed the capped problem at its default stopping rule, the solver reports "optimal" and the returned weights fail the independent check; reported beside it, the status at the 1e-9 rule the robust problem runs at and at the 1e-12 rule of the single-covariance problem; (c) the optimiser's full routine returns a portfolio that passes every check, with the cap raised to the smallest feasible turnover (within the rule's margin of 2e-6) and the month marked as relaxed.

Two repairs were tried, and the second is the one that holds. The first was to write the turnover cap differently. The cap can be written with absolute values (half the sum of |w - w0| at most 2%) or with a purchase p and a sale n per industry (w - w0 = p - n, half the sum of p + n at most 2%; an industry that goes from 7.0% to 6.4% has p = 0 and n = 0.6 points, and p + n is the 0.6 the absolute value gives). The two mean the same thing, and the rerun shows that the second way also comes back "optimal" at the default stopping rule, with different wrong weights (summing to 0.961 and trading 21.9%): rewriting the constraint changed which wrong weights came back and left the wrong status word in place, so it protects nothing. The second repair is the order of the two steps described at the start of this notebook. Both sets of wrong weights came from a problem that had no solution, and whether the capped problem has a solution can be known before it is posed: the smallest turnover the other constraints allow is a linear program, a problem with a linear objective and linear constraints, which the solver answers reliably, and comparing that turnover with the cap says whether the capped problem is feasible. The optimiser runs that program first in every month with a cap, raises the cap when that turnover is above it, and only then poses the tracking-error problem, so the solver never sees a capped problem without a solution. What this does not do is make the check unnecessary: on a feasible problem a solver can still return weights that are off by a little (at its default stopping rule, by 1e-4 against an independent solver's, which is why the stopping rule was tightened), so the check of every returned portfolio stays as the backstop. D5c runs the full routine.

In [ ]:
# Added after the first run of notebook 12 (6 October 2026): D5, the solver's status against the independent check at 1984-07.
import warnings
from bp import evaluation as E
d5_month = pd.Period(C.OPT_D5_MONTH, "M")
d5_months = pd.period_range(C.COV_EVAL_START, d5_month, freq="M")
Sig5 = {m: [CV.estimate(*v, m, P)[0] for v in CV.EVALUATION_VARIANTS] for m in d5_months}
path5 = E.run_path(d5_months, Sig5, weights, returns_m, C.CONSTRAINT_SET_TILT_ONLY, tilt_mask, None, robust=True, keep_weights=True)
prev = d5_months[-2]
w0 = O.drifted_weights(path5.loc[prev, "weights"], returns_m.loc[prev].to_numpy())
b = weights.loc[d5_month].to_numpy()
b_drift = O.drifted_weights(weights.loc[prev].to_numpy(), returns_m.loc[prev].to_numpy())
lower, upper = O.bounds(b, C.CONSTRAINT_SET_TILT_ONLY, tilt_mask, np.zeros(len(b), bool), C.ACTIVE_WEIGHT_BOUND, C.TILT_MAX_SHARE_OF_BENCHMARK)
Ls = [O.cholesky_factor(S) for S in Sig5[d5_month]]
cap = C.TURNOVER_CAP_MONTHLY_ONE_WAY

# (a) the smallest feasible one-way turnover, against the cap
w_t, prob_t = O._build(Ls, b, lower, upper, w0, None, objective="turnover")
prob_t.solve(solver=C.OPT_SOLVER, **C.OPT_SOLVER_OPTIONS_ROBUST)
min_turnover = float(prob_t.value)
d5a = min_turnover > cap
print(f"D5a: at {d5_month} the index moved {O.one_way_turnover(b, b_drift):.3f} one way; the smallest one-way turnover that satisfies the tilt and the active bound is {min_turnover:.4f}, "
      f"above the cap of {cap}: {'yes' if d5a else 'no'}, so the month is {'infeasible' if d5a else 'feasible'} under the cap: {'MET' if d5a else 'NOT MET'}")

# (b) the solver on the capped problem, at its default stopping rule and at the tight one, status against the independent check
def raw_solve(options):
    w_raw, prob_raw = O._build(Ls, b, lower, upper, w0, cap)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob_raw.solve(solver=C.OPT_SOLVER, **options)
    x = O._clean(w_raw.value)
    ok = len(x) == len(b) and O.feasible(x, b, lower, upper, w0, cap, C.OPT_TOL)
    return prob_raw.status, x, ok
status_default, x_default, ok_default = raw_solve({})
d5b = status_default == "optimal" and not ok_default
print(f"D5b: at the solver's default stopping rule (1e-8) the status is '{status_default}'"
      + (f", the returned weights sum to {x_default.sum():.4f} and trade {O.one_way_turnover(x_default, w0):.4f} one way, and the independent check {'passes' if ok_default else 'fails'}" if len(x_default) == len(b) else ", and no weights are returned")
      + f": {'MET' if d5b else 'NOT MET'}")
for label, options in (("1e-9, the robust problem's rule", C.OPT_SOLVER_OPTIONS_ROBUST), ("1e-12, the single-covariance rule", C.OPT_SOLVER_OPTIONS)):
    status_r, x_r, ok_r = raw_solve(options)
    print(f"     at the stopping rule {label}: status '{status_r}'" + (f", weights summing to {x_r.sum():.4f}, one-way turnover {O.one_way_turnover(x_r, w0):.4f}, the independent check {'passes' if ok_r else 'fails'}" if len(x_r) == len(b) else ", no weights returned"))

# (c) the guarded routine: feasibility test, relaxation to the smallest feasible turnover, every check passed
sol5 = O.solve_robust(Sig5[d5_month], b, C.CONSTRAINT_SET_TILT_ONLY, w_prev=w0, tilt_mask=tilt_mask)
d5c = bool(sol5.relaxed) and abs(sol5.turnover_cap_used - min_turnover) <= 2e-6 and O.feasible(sol5.weights, b, lower, upper, w0, sol5.turnover_cap_used, C.OPT_TOL)
print(f"D5c: the guarded routine relaxes the cap to {sol5.turnover_cap_used:.6f} (smallest feasible {min_turnover:.6f}), status '{sol5.status}', weights summing to {sol5.weights.sum():.6f}, "
      f"one-way turnover {sol5.turnover:.4f}, relaxed {sol5.relaxed}, every check passed: {'MET' if d5c else 'NOT MET'}")

## Save

The report table, the speed-limit path, the fixes table and the robust portfolio's table are written to `outputs/`, with the vintage in the file name, as a record.

In [ ]:
os.makedirs(C.OUTPUT_DIR, exist_ok=True)
report.to_csv(f"{C.OUTPUT_DIR}/optimiser_report_{vintage}.csv", float_format="%.6f")
path.to_csv(f"{C.OUTPUT_DIR}/optimiser_speed_limit_{vintage}.csv", float_format="%.6f")
fixes.to_csv(f"{C.OUTPUT_DIR}/optimiser_fixes_{vintage}.csv", float_format="%.6f")
rob_table.to_csv(f"{C.OUTPUT_DIR}/optimiser_robust_{vintage}.csv", float_format="%.6f")
print("written:", sorted(f for f in os.listdir(C.OUTPUT_DIR) if f.startswith("optimiser")))

## What this notebook established, and what could be wrong

| Expectation | Result | Verdict |
|---|---|---|
| 1. Count first | 686 months of benchmark weights summing to one, 685 months of drifted weights summing to one, Coal, Oil and Util present, 28 specifications | holds |
| 2a. Under C0 and C1 the benchmark is the answer | forecast tracking error below 1e-6 a year in 15 of 15 month-estimator pairs | MET |
| 2b. Under C3 the three tilts bind and the tracking error is positive | 15 of 15 | MET |
| 2c. Tracking error does not fall from C0 to C1 to C2, and rises from C2 to C3 | 15 of 15 and 15 of 15 | MET |
| 3. The tilt costs less than 100 basis points a year at the last month | 35.1 basis points (sample covariance, daily data) | MET |
| 4. Where the tilt's weight goes, effective number, binding constraints | weight to low-beta industries (Other, Food, Telcm, Smoke) and to the commodity-linked ones (Agric, Steel); effective number 18.4 to 24.2 in 1979-07, 10.2 to 10.3 in 2026-08; under C3 the three tilts and the relaxed cap bind | reported |
| 5. The cap as a speed limit from 1/N | first rebalance trades 0.0200 one way and leaves 708 basis points; C1 reaches the benchmark at once | MET |
| D1a. The market part of the tilt's active variance is below 5% in all 15 pairs | 14 of 15; the monthly sample covariance at 2004-11 leaves 5.4% (portfolio beta 1.007) | NOT MET |
| D2a. A beta-neutral constraint raises the tracking error by less than 1 basis point in all 15 pairs | 11 of 15; 3.4 and 5.6 basis points for the monthly sample covariance, 1.2 for the sample and shrinkage estimators on daily data at 2026-08 | NOT MET |
| D3. Excluding tobacco from the hedge (guess written before the code: a rise below 2 basis points) | rises of 1.3 to 6.5 basis points, 2% to 13% of the tilt's cost; below 2 in 4 of 15 pairs | reported; the guess was wrong |
| D4a. The daily estimators agree with each other on the hedge more than with the monthly sample covariance | 3 of 3 months (daily-daily mean rank correlation 0.58 to 0.68 against daily-monthly 0.54 to 0.63) | MET |
| D4b, D4c. Cross-evaluation and the absorbing industries by month | one estimator judges another's portfolio 1 to 5 basis points worse than its own; the absorbing industries change entirely across the three months | reported |
| F1. Under the mandate's C3 the beta is one and Smoke and Guns are at zero | 15 of 15 | MET |
| F2. The mandate's C3 costs more than the tilt alone and less than 15 basis points more | 13 of 15; the daily estimators pay 1.9 to 6.9 basis points, the monthly sample covariance 18.6 and 25.4 | NOT MET for the monthly covariance |
| F3. The robust portfolio's worst case is at most every single portfolio's worst case, and its excess under each covariance at most the largest single excess | 3 of 3 months; robust worst case 113, 64 and 42 basis points against 125, 67 and 42 for the best single portfolio; mean excess 0.1 to 1.1 basis points against single excesses of up to 21 | MET |
| D5. The solver's status against the independent check at 1984-07 (added 6 October) | the month is infeasible under the cap (smallest feasible turnover 0.0201, the index moved 4.3%); at the default and the 1e-9 stopping rules the status is "optimal" with weights summing to 0.961 and trading 22% one way, which fail the check; at 1e-12 the status is "infeasible"; the guarded routine relaxes the cap to 0.0201 and passes every check | MET, 3 of 3 |

Expectation 1 holds: the benchmark weights of notebook 08 sum to one in each of the 686 months from 1969-07, the drifted weights exist for the 685 months that follow a month and sum to one, Coal, Oil and Util are columns of the file, and the rolling evaluation of notebook 12 has 28 specifications, five estimator variants, the robust portfolio and RiskMetrics, times four constraint sets.

Expectation 2, the known answers, holds in every one of the 15 month-estimator pairs. Under long-only (C0) and with the active weight bound (C1) the solution is the benchmark itself, with a forecast tracking error of zero, because the benchmark satisfies both constraints and nothing beats zero. Under C3 the three tilted industries sit at exactly half their benchmark weight: the tilt is the only constraint that forces the portfolio off the benchmark, and a constraint that forces a deviation binds. The tracking error never falls from C0 to C1 to C2, which is a check of the solver (each set is the previous set plus one constraint), and it rises from C2 to C3 in every pair, which is a fact about the data: the freedom the tilt gives the three industries from the active bound is worth nothing when they are forced below it.

Expectation 3, the cost of the tilt, holds: 35 basis points a year at 2026-08 with the sample covariance on daily data, against the enhanced-indexing limit of 100. The cost depends on how much the tilt removes. In 1979-07 Oil was 15.1% of the benchmark and Utilities 8.0%, so the tilt took 11.7 percentage points out of the portfolio and cost 47 to 55 basis points with the daily estimators (108 with the sample covariance of 120 monthly returns); in 2004-11 it took 4.7 points and cost 49 to 56; in 2026-08 it takes 2.8 points and costs 33 to 36. Per percentage point removed, the cost rose from 4.6 basis points in 1979 to 12.7 in 2026: a concentrated benchmark (effective number of holdings 10.2 against 18.4) offers fewer substitutes of the right kind. The five estimators agree within 2 basis points in 2026-08 and within 8 in 1979-07 among the daily ones; the monthly sample covariance stands apart in 1979-07 (108 against 47 to 55) and is the lowest in 2004-11 and 2026-08.

Expectation 4, where the weight goes. The pre-run reasoning said the optimiser would put the removed weight in the industries that move most with Oil and Utilities. That is half of it. Oil and Utilities have betas to the benchmark of 0.41 and 0.33, so removing 2.8 points of them leaves a portfolio with a beta above one, and the first thing the optimiser does is restore the beta: the weight goes to Other (beta 0.47), Food (0.12), Telcm (0.39) and Smoke (0.04), and the portfolio's beta ends at 1.004. Among the low-beta industries it prefers those whose movement after the market is taken out is correlated with the removed ones: across the 46 untilted industries the rank correlation between the active weight and the residual correlation with Oil is 0.57 and with Utilities 0.43, against -0.29 with the beta, -0.13 with the industry's own volatility and 0.04 with its benchmark weight. The commodity-linked industries (Agric, Steel, FabPr, Gold) receive weight too, and little of it, because their own volatility is 25% to 42% a year. Only 11 of the 46 untilted industries receive any weight, because the first month's cap is relaxed to the smallest feasible turnover, so the portfolio buys exactly 2.8 points and places them where each unit traded hedges most. The effective number of holdings is the benchmark's under C0 to C2 (18.4 in 1979-07, 18.6 in 2004-11, 10.2 in 2026-08) and rises under C3 in 1979-07 to about 24, because the tilt spreads a 15% industry's weight over others; in 2026-08 it barely moves. The comparison with free trading separates the tilt from the transition: with free trading the same tilt costs 0.2 to 4 basis points less with the daily estimators and 5 to 17 less with the monthly sample covariance, which trades two to three times as much (0.24 against 0.12 in 1979-07) and holds fewer industries; that is the pattern of notebook 10 again, a noisy covariance sees hedges that a less noisy one does not, and notebook 12 will say whether those trades pay in realised tracking error.

Expectation 5, the cap as a speed limit, holds. From 1/N at 2026-08 the one-way distance to the benchmark is 0.528, which is 26 months of trading at 2% if every trade reduced the distance. Under C1 the first rebalance trades the whole 0.528 and lands on the benchmark; under long-only plus the cap the first rebalance trades exactly 0.0200 and leaves a forecast tracking error of 708 basis points, which falls to 472 after six rebalances, 254 after twelve and 28 after twenty-four, when the remaining distance is 0.078. The optimiser does not reduce the distance fastest; it reduces the tracking error fastest, which means trading the industries whose active weights contribute most to the deviation's variance first. Under C2 from the same start the active bound is hard and 1/N sits 0.387 of one-way turnover outside it, so the cap is relaxed to 0.387 and the portfolio lands inside the bound in one month with a forecast tracking error of 95 basis points.

**The practical problem behind the hedge (D1 to D4).** A tilt is also a factor bet: removing low-beta industries leaves a high-beta portfolio, and the optimiser's hedge depends on the covariance it is given. D1 measured how much of the tilt's forecast risk is a market bet after the optimiser has done its work: under 1% of the active variance for the four daily estimators at 1979-07 and 2004-11, 0.3% to 3% at 2026-08, and 2.3% and 5.4% for the monthly sample covariance at 1979-07 and 2004-11, where the portfolio's beta ends at 1.010 and 1.007. The expectation (below 5% in every pair) fails in that one pair, and the failure is informative: a noisy covariance sees cheap hedges in its noise and leaves a beta bet standing that a less noisy one removes. D2 imposed beta neutrality as a constraint: it costs the daily estimators 0.0 to 1.2 basis points and the monthly sample covariance 3.4 and 5.6, so the expectation (below 1 basis point in every pair) fails in four pairs, for the same reason. The lesson stands with a qualification: with a tracking-error objective the optimiser restores the beta on its own when the covariance is estimated from daily data; with 120 monthly observations it leaves a beta of 1.007 to 1.010 standing, and a manager using such a matrix should impose the constraint. D3 priced the mandate problem: at 2026-08 the hedge buys tobacco (Smoke, a low-beta industry) to replace Utilities, which a sustainability mandate that underweights carbon would refuse; excluding Smoke from the hedge costs 2.4 to 3.2 basis points at 2026-08, 1.3 to 1.6 at 1979-07 and 5.5 to 6.5 at 2004-11, that is 2% to 13% of the tilt's cost. The guess written before the code (below 2 basis points) was wrong in 11 of 15 pairs; the next-best low-beta substitutes are further away than the reasoning written before the code assumed. D4 measured how much the hedge is a property of the estimator. The four daily estimators agree with each other on the active weights more than with the monthly sample covariance (expectation met), and the agreement is partial: the sample covariance and shrinkage are the same hedge (rank correlation 0.99 to 1.00, because the shrinkage intensity on daily data is 0.012), the factor model agrees with the sample covariance at 0.45 to 0.53, and the component model sits between. In the cross-evaluation at 2026-08, one estimator judges another's hedge 1 to 5 basis points worse than its own on a tilt that costs 33 to 36: a model risk of up to 13% of the number quoted. And the industries that absorb the tilt change with the month: Fin, Telcm and Food in 1979-07, Drugs, Mach and Banks in 2004-11, Other, Food and Telcm in 2026-08. For a manager this says four things: check the factor exposures the optimiser creates and removes, because it hedges a beta bet silently; check the hedge against the mandate, because the covariance does not know what the mandate forbids; treat the forecast cost of a tilt as model-dependent to within about a tenth; and expect the hedge to be rebuilt as the covariance moves, which is turnover the cap will ration. The first three are problems a manager fixes in the design, and the design was changed accordingly.

**The fixes.** Beta neutrality and the exclusions of Smoke and Guns joined C3, and the robust portfolio joined the variants. Under the mandate's C3 every portfolio has a beta of exactly one and holds no tobacco or weapons (F1). The mandate costs the daily estimators 1.9 to 2.2 basis points more than the tilt alone at 1979-07, 5.6 to 6.9 at 2004-11 and 4.5 to 6.6 at 2026-08, that is 4% to 19% of the tilt's cost; the monthly sample covariance pays 25.4 and 18.6 basis points at the first two months and 0.6 at the last, so F2 (below 15 everywhere) fails for that covariance, which is the covariance whose hedges the fixes were most needed against: a matrix that saw cheap hedges in its noise is the one that loses most when the hedges it liked are forbidden. The robust portfolio does what it was built for (F3): its worst forecast tracking error across the five covariances is 113, 64 and 42 basis points at the three months against 125, 67 and 42 for the best single-estimator portfolio, and under each covariance it sits 0.1 to 1.1 basis points above that covariance's own minimum on average, where the single-estimator portfolios sit up to 21, 13 and 7 above each other's. The compromise costs almost nothing and protects against the matrix being wrong. Under the mandate the weight at 2026-08 goes to Food, Other, Telcm, Soda and Agric, for the robust portfolio and for the sample covariance alike, and to nothing the mandate forbids.

**The solver check (D5).** At 1984-07 the index moved 4.3% one way and the smallest turnover that satisfies the tilt and the active bound is 2.01%, a hundredth of a percentage point above the cap, so the month is infeasible under the cap by that margin. Handed the capped problem, the solver reports "optimal" at its default stopping rule and at the 1e-9 rule, with weights that sum to 0.961 and trade 21.9% of the portfolio one way, eleven times the cap; at the 1e-12 rule it reports "infeasible"; in notebook 12's first run, with the cap written with absolute values, it reported "optimal" with weights summing to 1.0037. The independent check fails the returned weights in every one of these cases, and the optimiser's full routine, which finds the smallest feasible turnover first and poses the capped problem only with a cap at or above it, raises the cap to 2.01% and returns a portfolio that sums to one and trades 2.01%, every check passed. Two consequences for a manager. First, a solver's status is a statement about its own arithmetic and says nothing about the mandate; a portfolio is accepted only after a check that does not belong to the solver, which in a firm is the pre-trade compliance test of the order management system and here is `optimiser.check`. Second, infeasibility is settled in the design before any solver runs: the design says which constraint gives way (the cap, raised to the smallest feasible turnover) and counts the months it happens, so that the fund never trades a portfolio the solver invented; notebook 12 reports those months (1 to 4 of 566 per path).

**What this notebook does not settle.**

- Every tracking error here is a forecast at three months from the covariance used to build the portfolio, so a portfolio built from an estimator is judged by that estimator's own view of it. Realised tracking error month by month, and whether the choice of estimator matters once the constraints are on, are notebook 12's question.
- Whether underweighting Coal, Oil and Utilities cost or earned return over the sample is a question about returns, which the optimiser never sees; notebook 13's attribution answers it.
- The transition rule, raising the cap to the smallest feasible turnover, decides the first month under a tilt and any month in which the benchmark moved by more than 2%. How often that happens and what it costs is a count notebook 12 reports.
- The reasoning written before the run for expectation 3 was incomplete: the number was right, the mechanism named only co-movement, and the first mechanism is the restoration of the portfolio's beta. The expectation was met for a reason only partly foreseen. D1 to D4 were designed after the first part and two of them failed in the pairs that use the monthly sample covariance; the fixes were designed after D1 to D4, and one of their expectations (F2) failed for the same covariance; D5 was designed after notebook 12's first run. Each step was written down before its code, with its date, and the order of the sections is the order of the work.
- The fixes are tested here on forecasts at three months. Whether the mandate's C3 and the robust portfolio hold up in realised tracking error, month by month, with the turnover of rebuilding the hedge as the covariance moves, is notebook 12's question; the tilt-only set runs beside them there so that the cost of the fixes is measured on realised numbers.
- Beta neutrality is the only factor constraint. Exposures to the other Fama-French factors (size, value, profitability, investment, momentum) are left to the covariance; imposing them would need factor betas from the window, with their own estimation error, and is the natural extension if notebook 13's attribution shows that the tilt loads on one of them.
- The optimiser was checked against an independent solver and against a linear program on problems of six to eight assets, and on the 49 industries by the feasibility checks and the known answers of expectation 2; no independent solver was run on the 49-industry problems. The solver's own status is not relied on in any month: D5 shows it reporting "optimal" on a month with no feasible portfolio, and every returned portfolio is tested against every constraint before it is accepted.
- The active weight bound is 2 percentage points around each benchmark weight. A bound proportional to the benchmark weight is another common choice and was not tried; the design fixed one and the notebook reports it.